In [1]:
import os


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import dask.dataframe as dd
from dask import delayed, compute
from dask.diagnostics import ProgressBar
from joblib import Memory, cpu_count
from pandas.plotting import register_matplotlib_converters
from sklearn.cluster import DBSCAN
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from joblib import Parallel, delayed, Memory, cpu_count
from sklearn.manifold import TSNE
from dtaidistance import dtw

In [2]:
memory = Memory(location='cachedir', verbose=0)

In [3]:
cpu_count()

32

In [4]:
# 手动注册 Matplotlib 转换器
register_matplotlib_converters()

# 项目背景

# 数据导入

In [5]:
@memory.cache
def read_hhblock_dataset(file_path):
    # 获取文件夹内所有CSV文件的路径
    csv_files = [os.path.join(file_path, file) for file in os.listdir(file_path) if file.endswith('.csv')]

    # 读取每个CSV文件为DataFrame，并存入列表
    dataframes = [pd.read_csv(file) for file in csv_files]

    # 合并所有DataFrame为一个大的DataFrame
    hhblock_big_dataframe = pd.concat(dataframes, ignore_index=True)

    return hhblock_big_dataframe

In [6]:
@memory.cache
def data_import():
    informations_households = pd.read_csv('archive/informations_households.csv')
    daily_dataset = pd.read_csv('archive/daily_dataset.csv')
    halfhourly_block_0 = pd.read_csv('archive/halfhourly_dataset/halfhourly_dataset/block_0.csv')
    hhblock_0 = pd.read_csv('archive/hhblock_dataset/hhblock_dataset/block_0.csv')
    hhblock_big_dataframe = read_hhblock_dataset('archive\hhblock_dataset\hhblock_dataset')
    uk_bank_holidays = pd.read_csv('archive/uk_bank_holidays.csv')
    weather_daily_darksky = pd.read_csv('archive/weather_daily_darksky.csv')
    weather_hourly_darksky = pd.read_csv('archive/weather_hourly_darksky.csv')
    return informations_households, daily_dataset, halfhourly_block_0, hhblock_0, hhblock_big_dataframe, uk_bank_holidays, weather_daily_darksky, weather_hourly_darksky

In [7]:
%%time
informations_households, daily_dataset, halfhourly_block_0, hhblock_0, hhblock_big_dataframe, uk_bank_holidays, weather_daily_darksky, weather_hourly_darksky = data_import()

CPU times: total: 219 ms
Wall time: 1.11 s


In [8]:
# %prun -s cumulative -l 10 informations_households, daily_dataset, halfhourly_block_0, hhblock_0, hhblock_big_dataframe, uk_bank_holidays, weather_daily_darksky, weather_hourly_darksky = data_import()

# 解释性描述数据与可视化
- 数据查看与理解
- 数据清洗
- 分组可视化

In [9]:
GLOBAL_PLOT_LEVEL = 0  # 0: 不显示图形，1: 显示部分图形，2: 显示所有图形
def plot_level(level, function):
    if GLOBAL_PLOT_LEVEL >= level:
        function()

## informations_households.csv
informations_households.csv - 包含有关每个家庭的信息，如家庭ID、家庭类型、家庭的总电量消耗等。
数据查看与清洗

In [10]:
informations_households

,LCLid,stdorToU,Acorn,Acorn_grouped,file
0,MAC005492,ToU,ACORN-,ACORN-,block_0
1,MAC001074,ToU,ACORN-,ACORN-,block_0
2,MAC000002,Std,ACORN-A,Affluent,block_0
3,MAC003613,Std,ACORN-A,Affluent,block_0
4,MAC003597,Std,ACORN-A,Affluent,block_0
...,...,...,...,...,...
5561,MAC002056,Std,ACORN-U,ACORN-U,block_111
5562,MAC004587,Std,ACORN-U,ACORN-U,block_111
5563,MAC004828,Std,ACORN-U,ACORN-U,block_111
5564,MAC001704,ToU,ACORN-U,ACORN-U,block_111


In [11]:
display(informations_households.nunique())

LCLid            5566
stdorToU            2
Acorn              19
Acorn_grouped       5
file              112
dtype: int64

In [12]:
display(informations_households['Acorn'].value_counts())

Acorn
ACORN-E    1567
ACORN-Q     831
ACORN-F     684
ACORN-H     455
ACORN-L     342
ACORN-D     292
ACORN-G     205
ACORN-K     165
ACORN-A     157
ACORN-N     152
ACORN-C     151
ACORN-M     113
ACORN-J     112
ACORN-P     110
ACORN-O     103
ACORN-I      51
ACORN-U      49
ACORN-B      25
ACORN-        2
Name: count, dtype: int64

In [13]:
display(informations_households['Acorn_grouped'].value_counts())

Acorn_grouped
Affluent       2192
Adversity      1816
Comfortable    1507
ACORN-U          49
ACORN-            2
Name: count, dtype: int64

In [14]:
display(informations_households['file'].value_counts())

file
block_0      50
block_1      50
block_82     50
block_81     50
block_80     50
             ..
block_33     50
block_32     50
block_31     50
block_30     50
block_111    16
Name: count, Length: 112, dtype: int64

In [15]:
informations_households_filtered = informations_households[
    (informations_households['Acorn'] != 'ACORN-U') & (informations_households['Acorn'] != 'ACORN-')]
informations_households_filtered

,LCLid,stdorToU,Acorn,Acorn_grouped,file
2,MAC000002,Std,ACORN-A,Affluent,block_0
3,MAC003613,Std,ACORN-A,Affluent,block_0
4,MAC003597,Std,ACORN-A,Affluent,block_0
5,MAC003579,Std,ACORN-A,Affluent,block_0
6,MAC003566,Std,ACORN-A,Affluent,block_0
...,...,...,...,...,...
5512,MAC002185,Std,ACORN-Q,Adversity,block_110
5513,MAC002347,Std,ACORN-Q,Adversity,block_110
5514,MAC000088,ToU,ACORN-Q,Adversity,block_110
5515,MAC002331,Std,ACORN-Q,Adversity,block_110


In [16]:
display(informations_households_filtered['Acorn'].value_counts())

Acorn
ACORN-E    1567
ACORN-Q     831
ACORN-F     684
ACORN-H     455
ACORN-L     342
ACORN-D     292
ACORN-G     205
ACORN-K     165
ACORN-A     157
ACORN-N     152
ACORN-C     151
ACORN-M     113
ACORN-J     112
ACORN-P     110
ACORN-O     103
ACORN-I      51
ACORN-B      25
Name: count, dtype: int64

In [17]:
display(informations_households_filtered['Acorn_grouped'].value_counts())

Acorn_grouped
Affluent       2192
Adversity      1816
Comfortable    1507
Name: count, dtype: int64

In [18]:
valid_lclid = informations_households_filtered['LCLid'].unique()
display(pd.Series(valid_lclid))

0       MAC000002
1       MAC003613
2       MAC003597
3       MAC003579
4       MAC003566
          ...    
5510    MAC002185
5511    MAC002347
5512    MAC000088
5513    MAC002331
5514    MAC000318
Length: 5515, dtype: object

In [19]:
unique_acorn_values = informations_households_filtered[informations_households_filtered['Acorn_grouped'] == 'Affluent'][
    'Acorn'].unique()
display(pd.Series(unique_acorn_values))

0    ACORN-A
1    ACORN-B
2    ACORN-C
3    ACORN-D
4    ACORN-E
dtype: object

In [20]:
unique_acorn_values = \
    informations_households_filtered[informations_households_filtered['Acorn_grouped'] == 'Comfortable'][
        'Acorn'].unique()
display(pd.Series(unique_acorn_values))

0    ACORN-F
1    ACORN-G
2    ACORN-H
3    ACORN-I
4    ACORN-J
dtype: object

In [21]:
unique_acorn_values = \
    informations_households_filtered[informations_households_filtered['Acorn_grouped'] == 'Adversity']['Acorn'].unique()
display(pd.Series(unique_acorn_values))

0    ACORN-K
1    ACORN-L
2    ACORN-M
3    ACORN-N
4    ACORN-O
5    ACORN-P
6    ACORN-Q
dtype: object

In [22]:
missing_informations_households_values = informations_households_filtered.isnull().sum()
display(missing_informations_households_values)

LCLid            0
stdorToU         0
Acorn            0
Acorn_grouped    0
file             0
dtype: int64

## daily_dataset

### 数据查看与清洗

In [23]:
daily_dataset.head(1000)

,LCLid,day,energy_median,energy_mean,energy_max,energy_count,energy_std,energy_sum,energy_min
0,MAC000131,2011-12-15,0.4850,0.432045,0.868,22,0.239146,9.505,0.072
1,MAC000131,2011-12-16,0.1415,0.296167,1.116,48,0.281471,14.216,0.031
2,MAC000131,2011-12-17,0.1015,0.189812,0.685,48,0.188405,9.111,0.064
3,MAC000131,2011-12-18,0.1140,0.218979,0.676,48,0.202919,10.511,0.065
4,MAC000131,2011-12-19,0.1910,0.325979,0.788,48,0.259205,15.647,0.066
...,...,...,...,...,...,...,...,...,...
995,MAC000132,2012-06-20,0.1350,0.267854,1.208,48,0.238778,12.857,0.072
996,MAC000132,2012-06-21,0.1505,0.240729,0.954,48,0.242623,11.555,0.057
997,MAC000132,2012-06-22,0.1490,0.225021,1.193,48,0.231525,10.801,0.046
998,MAC000132,2012-06-23,0.2135,0.317583,0.875,48,0.241235,15.244,0.060


In [24]:
@ memory.cache
def process_daily_dataset(df):
    # 将'day'列转换为日期时间格式
    df['day'] = pd.to_datetime(df['day'])

    # 将'LCLid'列转换为分类数据类型
    df['LCLid'] = df['LCLid'].astype('category')
    
    # 清洗掉没有Acorn信息的数据
    df = df[df['LCLid'].isin(valid_lclid)]

    return df

In [25]:
filtered_daily_dataset = process_daily_dataset(daily_dataset)
filtered_daily_dataset.head(1000)

,LCLid,day,energy_median,energy_mean,energy_max,energy_count,energy_std,energy_sum,energy_min
0,MAC000131,2011-12-15,0.4850,0.432045,0.868,22,0.239146,9.505,0.072
1,MAC000131,2011-12-16,0.1415,0.296167,1.116,48,0.281471,14.216,0.031
2,MAC000131,2011-12-17,0.1015,0.189812,0.685,48,0.188405,9.111,0.064
3,MAC000131,2011-12-18,0.1140,0.218979,0.676,48,0.202919,10.511,0.065
4,MAC000131,2011-12-19,0.1910,0.325979,0.788,48,0.259205,15.647,0.066
...,...,...,...,...,...,...,...,...,...
995,MAC000132,2012-06-20,0.1350,0.267854,1.208,48,0.238778,12.857,0.072
996,MAC000132,2012-06-21,0.1505,0.240729,0.954,48,0.242623,11.555,0.057
997,MAC000132,2012-06-22,0.1490,0.225021,1.193,48,0.231525,10.801,0.046
998,MAC000132,2012-06-23,0.2135,0.317583,0.875,48,0.241235,15.244,0.060


In [26]:
filtered_daily_dataset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3480100 entries, 0 to 3510432
Data columns (total 9 columns):
 #   Column         Dtype         
---  ------         -----         
 0   LCLid          category      
 1   day            datetime64[ns]
 2   energy_median  float64       
 3   energy_mean    float64       
 4   energy_max     float64       
 5   energy_count   int64         
 6   energy_std     float64       
 7   energy_sum     float64       
 8   energy_min     float64       
dtypes: category(1), datetime64[ns](1), float64(6), int64(1)
memory usage: 245.6 MB


### 缺失值处理

In [27]:
missing_filtered_daily_dataset_values = filtered_daily_dataset.isnull().sum()
display(missing_filtered_daily_dataset_values)

LCLid                0
day                  0
energy_median       30
energy_mean         30
energy_max          30
energy_count         0
energy_std       11214
energy_sum          30
energy_min          30
dtype: int64

In [28]:
rows_with_missing_values_specific_columns = filtered_daily_dataset[
    filtered_daily_dataset[['energy_std']].isnull().any(axis=1)]
rows_with_missing_values_specific_columns

,LCLid,day,energy_median,energy_mean,energy_max,energy_count,energy_std,energy_sum,energy_min
806,MAC000131,2014-02-28,0.075,0.075,0.075,1,NaN,0.075,0.075
1613,MAC000132,2014-02-28,0.049,0.049,0.049,1,NaN,0.049,0.049
2434,MAC000221,2014-02-28,0.592,0.592,0.592,1,NaN,0.592,0.592
3255,MAC000228,2014-02-28,0.039,0.039,0.039,1,NaN,0.039,0.039
4076,MAC000234,2014-02-28,0.071,0.071,0.071,1,NaN,0.071,0.071
...,...,...,...,...,...,...,...,...,...
3507356,MAC004926,2014-02-28,0.033,0.033,0.033,1,NaN,0.033,0.033
3508127,MAC004932,2014-02-28,0.177,0.177,0.177,1,NaN,0.177,0.177
3508898,MAC004937,2014-02-28,0.084,0.084,0.084,1,NaN,0.084,0.084
3509666,MAC004965,2014-02-28,0.618,0.618,0.618,1,NaN,0.618,0.618


In [29]:
rows_with_missing_values_specific_columns = filtered_daily_dataset[
    filtered_daily_dataset[['energy_median', 'energy_mean']].isnull().any(axis=1)]
rows_with_missing_values_specific_columns

,LCLid,day,energy_median,energy_mean,energy_max,energy_count,energy_std,energy_sum,energy_min
37231,MAC000410,2012-12-18,NaN,NaN,NaN,0,NaN,NaN,NaN
129012,MAC005560,2012-12-19,NaN,NaN,NaN,0,NaN,NaN,NaN
138461,MAC002110,2012-12-18,NaN,NaN,NaN,0,NaN,NaN,NaN
205939,MAC001065,2012-12-18,NaN,NaN,NaN,0,NaN,NaN,NaN
293243,MAC001229,2012-12-18,NaN,NaN,NaN,0,NaN,NaN,NaN
295223,MAC001278,2012-12-18,NaN,NaN,NaN,0,NaN,NaN,NaN
692446,MAC001478,2013-03-08,NaN,NaN,NaN,0,NaN,NaN,NaN
867057,MAC005558,2012-12-19,NaN,NaN,NaN,0,NaN,NaN,NaN
902532,MAC000393,2012-12-18,NaN,NaN,NaN,0,NaN,NaN,NaN
913986,MAC002014,2012-12-18,NaN,NaN,NaN,0,NaN,NaN,NaN


### 分组可视化

In [30]:
def plot_column_histogram(df, column_name):
    plt.figure(figsize=(10, 6))
    sns.histplot(df[column_name], kde=True)
    plt.title(f'Histogram of {column_name}')
    plt.xlabel(column_name)
    plt.ylabel('Frequency')
    plt.show()

In [31]:
plot_level(1, lambda: plot_column_histogram(filtered_daily_dataset, 'energy_median'))

In [32]:
columns_all = ['energy_median', 'energy_mean', 'energy_max', 'energy_std', 'energy_sum', 'energy_min']
columns_without_sum = ['energy_median', 'energy_mean', 'energy_max', 'energy_std', 'energy_min']
columns_without_min = ['energy_median', 'energy_mean', 'energy_max', 'energy_std', 'energy_sum']

In [33]:
def plot_column_histograms(df, column_names):
    plt.figure(figsize=(10, 6))
    for column_name in column_names:
        sns.histplot(df[column_name], kde=True, label=column_name)
    plt.title(f'Histogram of {" vs ".join(column_names)}')
    plt.xlabel('Value')
    plt.ylabel('Frequency')
    plt.xscale('log')  # 使用对数尺度
    plt.legend()
    plt.show()

In [34]:
plot_level(1, lambda: plot_column_histograms(filtered_daily_dataset, columns_all))
plot_level(1, lambda: plot_column_histograms(filtered_daily_dataset, columns_without_sum))
plot_level(1, lambda: plot_column_histograms(filtered_daily_dataset, columns_without_min))

In [35]:
def plot_multiple_column_histograms(df, columns):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    for i, column in enumerate(columns):
        sns.histplot(df[column], kde=True, ax=axes[i])
        axes[i].set_title(f'Histogram of {column}')
        axes[i].set_xlabel(column)
        axes[i].set_ylabel('Frequency')
    plt.tight_layout()
    plt.show()

In [36]:
plot_level(1, lambda: plot_multiple_column_histograms(filtered_daily_dataset, columns_all))

In [37]:
def plot_columns_boxplot(df, columns):
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df[columns])
    plt.xticks(rotation=45)
    plt.title('Box plot of Energy Consumption Statistics')
    plt.show()

In [38]:
plot_level(1, lambda: plot_columns_boxplot(filtered_daily_dataset, columns_all))

In [39]:
plot_level(1, lambda: plot_columns_boxplot(filtered_daily_dataset, columns_without_sum))

In [40]:
def plot_column_difference(df, column1, column2):
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=column1, y=column2, data=df)
    plt.title(f'Scatter plot of {column1} vs. {column2}')
    plt.xlabel(column1)
    plt.ylabel(column2)
    plt.show()

In [41]:
plot_level(1, lambda: plot_column_difference(filtered_daily_dataset, 'energy_mean', 'energy_median'))

In [42]:
plot_level(1, lambda: plot_column_difference(filtered_daily_dataset, 'energy_mean', 'energy_max'))

In [43]:
plot_level(1, lambda: plot_column_difference(filtered_daily_dataset, 'energy_mean', 'energy_std'))

In [44]:
plot_level(1, lambda: plot_column_difference(filtered_daily_dataset, 'energy_mean', 'energy_sum'))

In [45]:
plot_level(1, lambda: plot_column_difference(filtered_daily_dataset, 'energy_mean', 'energy_min'))

In [46]:
plot_level(1, lambda: plot_column_difference(filtered_daily_dataset, 'energy_median', 'energy_max'))

In [47]:
plot_level(1, lambda: plot_column_difference(filtered_daily_dataset, 'energy_median', 'energy_count'))

In [48]:
def plot_yearly_energy_trend(df, stat_col='energy_mean'):
    # 设置图形的大小
    plt.figure(figsize=(14, 8))

    # 获取年份列表
    years = df['day'].dt.year.unique()
    for year in sorted(years):
        # 选择当前年份的数据
        df_year = df[df['day'].dt.year == year]

        # 按'day'分组并计算每一天的平均用电量
        daily_avg = df_year.groupby('day')[stat_col].mean()

        # 重置索引，确保能够按时间序列绘图
        daily_avg = daily_avg.reset_index()

        # 绘制当前年份的日平均用电量趋势
        plt.plot(daily_avg['day'], daily_avg[stat_col], label=str(year))

    # 设置图例
    plt.legend(title="Year")

    # 设置标题和坐标轴标签
    plt.title(f'Yearly Trend of Daily Average {stat_col.capitalize()}')
    plt.xlabel('Day of Year')
    plt.ylabel(f'Average {stat_col.capitalize()}')

    # 显示网格
    plt.grid(True)

    # 显示图形
    plt.show()

# 假设filtered_daily_dataset是你的DataFrame
# plot_yearly_energy_trend(filtered_daily_dataset)


In [49]:
plot_level(1, lambda: plot_yearly_energy_trend(filtered_daily_dataset))

## halfhourly_dataset

In [50]:
halfhourly_block_0

,LCLid,tstp,energy(kWh/hh)
0,MAC000002,2012-10-12 00:30:00.0000000,0
1,MAC000002,2012-10-12 01:00:00.0000000,0
2,MAC000002,2012-10-12 01:30:00.0000000,0
3,MAC000002,2012-10-12 02:00:00.0000000,0
4,MAC000002,2012-10-12 02:30:00.0000000,0
...,...,...,...
1222665,MAC005492,2014-02-27 22:00:00.0000000,0.182
1222666,MAC005492,2014-02-27 22:30:00.0000000,0.122
1222667,MAC005492,2014-02-27 23:00:00.0000000,0.14
1222668,MAC005492,2014-02-27 23:30:00.0000000,0.192


In [51]:
display(halfhourly_block_0.nunique())

LCLid                50
tstp              39292
energy(kWh/hh)     5022
dtype: int64

In [52]:
halfhourly_block_0.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1222670 entries, 0 to 1222669
Data columns (total 3 columns):
 #   Column          Non-Null Count    Dtype 
---  ------          --------------    ----- 
 0   LCLid           1222670 non-null  object
 1   tstp            1222670 non-null  object
 2   energy(kWh/hh)  1222670 non-null  object
dtypes: object(3)
memory usage: 28.0+ MB


## hhblock_dataset

### 数据查看与清洗

In [53]:
hhblock_0.head(1000)

,LCLid,day,hh_0,hh_1,hh_2,hh_3,hh_4,hh_5,hh_6,hh_7,...,hh_38,hh_39,hh_40,hh_41,hh_42,hh_43,hh_44,hh_45,hh_46,hh_47
0,MAC000002,2012-10-13,0.263,0.269,0.275,0.256,0.211,0.136,0.161,0.119,...,0.918,0.278,0.267,0.239,0.230,0.233,0.235,0.188,0.259,0.250
1,MAC000002,2012-10-14,0.262,0.166,0.226,0.088,0.126,0.082,0.123,0.083,...,1.075,0.956,0.821,0.745,0.712,0.511,0.231,0.210,0.278,0.159
2,MAC000002,2012-10-15,0.192,0.097,0.141,0.083,0.132,0.070,0.130,0.074,...,1.164,0.249,0.225,0.258,0.260,0.334,0.299,0.236,0.241,0.237
3,MAC000002,2012-10-16,0.237,0.237,0.193,0.118,0.098,0.107,0.094,0.109,...,0.966,0.172,0.192,0.228,0.203,0.211,0.188,0.213,0.157,0.202
4,MAC000002,2012-10-17,0.157,0.211,0.155,0.169,0.101,0.117,0.084,0.118,...,0.223,0.075,0.230,0.208,0.265,0.377,0.327,0.277,0.288,0.256
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,MAC000246,2013-04-15,0.702,0.764,0.356,0.041,0.055,0.050,0.039,0.093,...,0.698,2.153,2.850,1.241,1.039,1.209,0.860,0.457,0.802,0.742
996,MAC000246,2013-04-16,0.640,0.095,0.748,0.693,0.302,0.024,0.029,0.072,...,0.820,2.551,2.907,2.186,2.239,1.239,1.003,0.685,0.887,0.550
997,MAC000246,2013-04-17,0.600,0.776,0.453,0.064,0.057,0.045,0.058,0.024,...,0.834,0.364,1.113,1.081,0.353,0.731,0.836,0.701,0.233,0.171
998,MAC000246,2013-04-18,0.134,0.120,0.277,0.235,0.148,0.749,0.504,0.073,...,0.121,1.189,1.935,2.145,1.430,1.486,1.099,0.960,0.761,0.215


In [54]:
hhblock_0.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25286 entries, 0 to 25285
Data columns (total 50 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   LCLid   25286 non-null  object 
 1   day     25286 non-null  object 
 2   hh_0    25286 non-null  float64
 3   hh_1    25286 non-null  float64
 4   hh_2    25286 non-null  float64
 5   hh_3    25286 non-null  float64
 6   hh_4    25286 non-null  float64
 7   hh_5    25286 non-null  float64
 8   hh_6    25286 non-null  float64
 9   hh_7    25286 non-null  float64
 10  hh_8    25286 non-null  float64
 11  hh_9    25286 non-null  float64
 12  hh_10   25286 non-null  float64
 13  hh_11   25286 non-null  float64
 14  hh_12   25286 non-null  float64
 15  hh_13   25286 non-null  float64
 16  hh_14   25286 non-null  float64
 17  hh_15   25286 non-null  float64
 18  hh_16   25286 non-null  float64
 19  hh_17   25286 non-null  float64
 20  hh_18   25286 non-null  float64
 21  hh_19   25286 non-null  float64
 22

In [55]:
# 查看合并后的DataFrame信息
hhblock_big_dataframe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3469352 entries, 0 to 3469351
Data columns (total 50 columns):
 #   Column  Dtype  
---  ------  -----  
 0   LCLid   object 
 1   day     object 
 2   hh_0    float64
 3   hh_1    float64
 4   hh_2    float64
 5   hh_3    float64
 6   hh_4    float64
 7   hh_5    float64
 8   hh_6    float64
 9   hh_7    float64
 10  hh_8    float64
 11  hh_9    float64
 12  hh_10   float64
 13  hh_11   float64
 14  hh_12   float64
 15  hh_13   float64
 16  hh_14   float64
 17  hh_15   float64
 18  hh_16   float64
 19  hh_17   float64
 20  hh_18   float64
 21  hh_19   float64
 22  hh_20   float64
 23  hh_21   float64
 24  hh_22   float64
 25  hh_23   float64
 26  hh_24   float64
 27  hh_25   float64
 28  hh_26   float64
 29  hh_27   float64
 30  hh_28   float64
 31  hh_29   float64
 32  hh_30   float64
 33  hh_31   float64
 34  hh_32   float64
 35  hh_33   float64
 36  hh_34   float64
 37  hh_35   float64
 38  hh_36   float64
 39  hh_37   float64
 40  

In [56]:
hhblock_big_dataframe.head(1000)

,LCLid,day,hh_0,hh_1,hh_2,hh_3,hh_4,hh_5,hh_6,hh_7,...,hh_38,hh_39,hh_40,hh_41,hh_42,hh_43,hh_44,hh_45,hh_46,hh_47
0,MAC000002,2012-10-13,0.263,0.269,0.275,0.256,0.211,0.136,0.161,0.119,...,0.918,0.278,0.267,0.239,0.230,0.233,0.235,0.188,0.259,0.250
1,MAC000002,2012-10-14,0.262,0.166,0.226,0.088,0.126,0.082,0.123,0.083,...,1.075,0.956,0.821,0.745,0.712,0.511,0.231,0.210,0.278,0.159
2,MAC000002,2012-10-15,0.192,0.097,0.141,0.083,0.132,0.070,0.130,0.074,...,1.164,0.249,0.225,0.258,0.260,0.334,0.299,0.236,0.241,0.237
3,MAC000002,2012-10-16,0.237,0.237,0.193,0.118,0.098,0.107,0.094,0.109,...,0.966,0.172,0.192,0.228,0.203,0.211,0.188,0.213,0.157,0.202
4,MAC000002,2012-10-17,0.157,0.211,0.155,0.169,0.101,0.117,0.084,0.118,...,0.223,0.075,0.230,0.208,0.265,0.377,0.327,0.277,0.288,0.256
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,MAC000246,2013-04-15,0.702,0.764,0.356,0.041,0.055,0.050,0.039,0.093,...,0.698,2.153,2.850,1.241,1.039,1.209,0.860,0.457,0.802,0.742
996,MAC000246,2013-04-16,0.640,0.095,0.748,0.693,0.302,0.024,0.029,0.072,...,0.820,2.551,2.907,2.186,2.239,1.239,1.003,0.685,0.887,0.550
997,MAC000246,2013-04-17,0.600,0.776,0.453,0.064,0.057,0.045,0.058,0.024,...,0.834,0.364,1.113,1.081,0.353,0.731,0.836,0.701,0.233,0.171
998,MAC000246,2013-04-18,0.134,0.120,0.277,0.235,0.148,0.749,0.504,0.073,...,0.121,1.189,1.935,2.145,1.430,1.486,1.099,0.960,0.761,0.215


In [57]:
def process_hhblock_dataset(df):
    # 将'day'列转换为日期时间格式
    df['day'] = pd.to_datetime(df['day'])

    # 将'LCLid'列转换为分类数据类型
    df['LCLid'] = df['LCLid'].astype('category')
    
    # 清洗掉没有Acorn信息的数据
    df = df[df['LCLid'].isin(valid_lclid)]

    return df

In [58]:
filtered_big_hhblock = process_hhblock_dataset(hhblock_big_dataframe)
filtered_big_hhblock.head(1000)

,LCLid,day,hh_0,hh_1,hh_2,hh_3,hh_4,hh_5,hh_6,hh_7,...,hh_38,hh_39,hh_40,hh_41,hh_42,hh_43,hh_44,hh_45,hh_46,hh_47
0,MAC000002,2012-10-13,0.263,0.269,0.275,0.256,0.211,0.136,0.161,0.119,...,0.918,0.278,0.267,0.239,0.230,0.233,0.235,0.188,0.259,0.250
1,MAC000002,2012-10-14,0.262,0.166,0.226,0.088,0.126,0.082,0.123,0.083,...,1.075,0.956,0.821,0.745,0.712,0.511,0.231,0.210,0.278,0.159
2,MAC000002,2012-10-15,0.192,0.097,0.141,0.083,0.132,0.070,0.130,0.074,...,1.164,0.249,0.225,0.258,0.260,0.334,0.299,0.236,0.241,0.237
3,MAC000002,2012-10-16,0.237,0.237,0.193,0.118,0.098,0.107,0.094,0.109,...,0.966,0.172,0.192,0.228,0.203,0.211,0.188,0.213,0.157,0.202
4,MAC000002,2012-10-17,0.157,0.211,0.155,0.169,0.101,0.117,0.084,0.118,...,0.223,0.075,0.230,0.208,0.265,0.377,0.327,0.277,0.288,0.256
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,MAC000246,2013-04-15,0.702,0.764,0.356,0.041,0.055,0.050,0.039,0.093,...,0.698,2.153,2.850,1.241,1.039,1.209,0.860,0.457,0.802,0.742
996,MAC000246,2013-04-16,0.640,0.095,0.748,0.693,0.302,0.024,0.029,0.072,...,0.820,2.551,2.907,2.186,2.239,1.239,1.003,0.685,0.887,0.550
997,MAC000246,2013-04-17,0.600,0.776,0.453,0.064,0.057,0.045,0.058,0.024,...,0.834,0.364,1.113,1.081,0.353,0.731,0.836,0.701,0.233,0.171
998,MAC000246,2013-04-18,0.134,0.120,0.277,0.235,0.148,0.749,0.504,0.073,...,0.121,1.189,1.935,2.145,1.430,1.486,1.099,0.960,0.761,0.215


In [59]:
filtered_big_hhblock.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3439414 entries, 0 to 3469351
Data columns (total 50 columns):
 #   Column  Dtype         
---  ------  -----         
 0   LCLid   category      
 1   day     datetime64[ns]
 2   hh_0    float64       
 3   hh_1    float64       
 4   hh_2    float64       
 5   hh_3    float64       
 6   hh_4    float64       
 7   hh_5    float64       
 8   hh_6    float64       
 9   hh_7    float64       
 10  hh_8    float64       
 11  hh_9    float64       
 12  hh_10   float64       
 13  hh_11   float64       
 14  hh_12   float64       
 15  hh_13   float64       
 16  hh_14   float64       
 17  hh_15   float64       
 18  hh_16   float64       
 19  hh_17   float64       
 20  hh_18   float64       
 21  hh_19   float64       
 22  hh_20   float64       
 23  hh_21   float64       
 24  hh_22   float64       
 25  hh_23   float64       
 26  hh_24   float64       
 27  hh_25   float64       
 28  hh_26   float64       
 29  hh_27   float64    

### 缺失值处理

In [60]:
missing_filtered_big_hhblock_values = filtered_big_hhblock.isnull().sum()
missing_filtered_big_hhblock_values

LCLid       0
day         0
hh_0        0
hh_1        0
hh_2        0
hh_3        0
hh_4        0
hh_5        0
hh_6        0
hh_7        0
hh_8        0
hh_9        0
hh_10       0
hh_11       0
hh_12       0
hh_13       0
hh_14       0
hh_15       0
hh_16       0
hh_17       0
hh_18       0
hh_19       2
hh_20       0
hh_21       0
hh_22       0
hh_23       0
hh_24       0
hh_25      21
hh_26       1
hh_27       0
hh_28       0
hh_29       0
hh_30    5410
hh_31       0
hh_32       0
hh_33       0
hh_34       0
hh_35       0
hh_36       1
hh_37       0
hh_38       0
hh_39       0
hh_40       0
hh_41       0
hh_42       0
hh_43       0
hh_44       0
hh_45       0
hh_46       0
hh_47       0
dtype: int64

In [61]:
@ memory.cache
def data_for_hhblock_plot():
    # 选择所有数值列用于分组聚合
    df_numeric = filtered_big_hhblock.iloc[:, 2:]  # 选择从第三列到最后的所有列
    
    # 按年分组求均值
    df_yearly_mean = df_numeric.groupby(filtered_big_hhblock['day'].dt.to_period('Y')).mean()
    
    # 按季度分组求均值
    df_quarterly_mean = df_numeric.groupby(filtered_big_hhblock['day'].dt.to_period('Q')).mean()
    
    # 按月分组求均值
    df_monthly_mean = df_numeric.groupby(filtered_big_hhblock['day'].dt.to_period('M')).mean()
    
    # 获取季度和月份的字符串表示，不考虑年份
    # 例如，将"2011Q4"转换为"Q4"，将2012-05转换为"M5"
    quarter_str = filtered_big_hhblock['day'].dt.to_period('Q').astype(str).str[-2:]
    month_str = filtered_big_hhblock['day'].dt.month.apply(lambda x: f"M{x}")
    
    # 按季度分组求均值，这里使用转换后的季度字符串
    df_year_quarterly_mean = df_numeric.groupby(quarter_str).mean()
    
    # 按月份分组求均值，这里使用转换后的月份字符串
    df_year_monthly_mean = df_numeric.groupby(month_str).mean()
    
    # 按星期几分组求均值
    df_dayofweek_mean = df_numeric.groupby(filtered_big_hhblock['day'].dt.dayofweek).mean()
    
    return df_yearly_mean, df_quarterly_mean, df_monthly_mean, df_year_quarterly_mean, df_year_monthly_mean, df_dayofweek_mean

In [62]:
df_yearly_mean, df_quarterly_mean, df_monthly_mean, df_year_quarterly_mean, df_year_monthly_mean, df_dayofweek_mean = data_for_hhblock_plot()

In [63]:
df_yearly_mean

,hh_0,hh_1,hh_2,hh_3,hh_4,hh_5,hh_6,hh_7,hh_8,hh_9,...,hh_38,hh_39,hh_40,hh_41,hh_42,hh_43,hh_44,hh_45,hh_46,hh_47
day,,,,,,,,,,,,,,,,,,,,,
2011,0.226201,0.24568,0.221275,0.195596,0.175225,0.162091,0.154699,0.144014,0.133944,0.133677,...,0.383679,0.379936,0.369906,0.355203,0.347197,0.335236,0.324993,0.304083,0.279174,0.253856
2012,0.159381,0.16862,0.151961,0.137585,0.126645,0.119919,0.115118,0.112078,0.111088,0.113655,...,0.310947,0.310509,0.306104,0.300896,0.291368,0.279294,0.257216,0.232182,0.204753,0.178719
2013,0.184745,0.16436,0.147354,0.134487,0.125518,0.120020,0.115712,0.113359,0.112772,0.115713,...,0.317786,0.317877,0.313502,0.308174,0.298632,0.286424,0.265282,0.241039,0.213262,0.186334
2014,0.226068,0.20113,0.177073,0.158040,0.143906,0.135155,0.129038,0.124343,0.120784,0.120477,...,0.381394,0.379103,0.367036,0.354726,0.341916,0.327858,0.308835,0.289947,0.259638,0.225511


In [64]:
df_quarterly_mean

,hh_0,hh_1,hh_2,hh_3,hh_4,hh_5,hh_6,hh_7,hh_8,hh_9,...,hh_38,hh_39,hh_40,hh_41,hh_42,hh_43,hh_44,hh_45,hh_46,hh_47
day,,,,,,,,,,,,,,,,,,,,,
2011Q4,0.226201,0.245680,0.221275,0.195596,0.175225,0.162091,0.154699,0.144014,0.133944,0.133677,...,0.383679,0.379936,0.369906,0.355203,0.347197,0.335236,0.324993,0.304083,0.279174,0.253856
2012Q1,0.209771,0.310967,0.277628,0.245669,0.216038,0.195507,0.181364,0.167980,0.157781,0.153169,...,0.386493,0.390015,0.380736,0.371728,0.357558,0.343285,0.322938,0.300625,0.268069,0.232011
2012Q2,0.137277,0.157365,0.142929,0.128968,0.118306,0.112503,0.108702,0.106239,0.106892,0.111569,...,0.260276,0.263451,0.266395,0.267539,0.264199,0.255828,0.231871,0.203206,0.175490,0.152413
2012Q3,0.130201,0.130736,0.119133,0.109516,0.103800,0.100794,0.098181,0.096950,0.097384,0.100871,...,0.252736,0.256613,0.256824,0.255322,0.247685,0.236615,0.214782,0.189460,0.165254,0.144934
2012Q4,0.189138,0.188218,0.168269,0.151895,0.138766,0.130182,0.124224,0.120703,0.118882,0.120549,...,0.376950,0.370414,0.358954,0.347841,0.334183,0.319673,0.298091,0.274766,0.245359,0.214207
2013Q1,0.244006,0.215267,0.191614,0.172602,0.158003,0.148439,0.141510,0.136939,0.134069,0.133747,...,0.407327,0.405880,0.394920,0.383283,0.371070,0.356727,0.337140,0.315500,0.282706,0.246222
2013Q2,0.153557,0.138017,0.125485,0.116116,0.110448,0.107476,0.104906,0.104176,0.105139,0.110761,...,0.264239,0.269302,0.272621,0.274390,0.267965,0.256550,0.231813,0.202638,0.175267,0.152055
2013Q3,0.140656,0.126481,0.115432,0.108487,0.104327,0.101624,0.099019,0.097829,0.098796,0.103469,...,0.243198,0.246473,0.247669,0.247277,0.240764,0.231540,0.210414,0.186189,0.162811,0.143303
2013Q4,0.200443,0.177387,0.156583,0.140434,0.128990,0.122256,0.117135,0.114223,0.112818,0.114616,...,0.356271,0.349581,0.338420,0.327281,0.314202,0.300340,0.281276,0.259391,0.231907,0.203483


In [65]:
df_monthly_mean

,hh_0,hh_1,hh_2,hh_3,hh_4,hh_5,hh_6,hh_7,hh_8,hh_9,...,hh_38,hh_39,hh_40,hh_41,hh_42,hh_43,hh_44,hh_45,hh_46,hh_47
day,,,,,,,,,,,,,,,,,,,,,
2011-11,0.199826,0.226474,0.211819,0.196941,0.193441,0.172685,0.155222,0.144552,0.127522,0.125133,...,0.316441,0.330333,0.317011,0.310348,0.302056,0.295026,0.294744,0.282011,0.256685,0.232770
2011-12,0.227123,0.246352,0.221606,0.195549,0.174588,0.161720,0.154681,0.143995,0.134168,0.133976,...,0.386030,0.381670,0.371755,0.356771,0.348776,0.336642,0.326051,0.304855,0.279961,0.254593
2012-01,0.223491,0.289421,0.257377,0.229763,0.203643,0.184150,0.174570,0.160749,0.151930,0.147932,...,0.420052,0.415183,0.401288,0.391381,0.375023,0.359044,0.339462,0.318275,0.285698,0.254078
2012-02,0.232255,0.349428,0.313940,0.281096,0.251662,0.228801,0.214398,0.199987,0.184847,0.177134,...,0.417492,0.417689,0.409205,0.401762,0.388464,0.373001,0.351558,0.328357,0.296102,0.254422
2012-03,0.188977,0.295700,0.263174,0.229830,0.198486,0.179001,0.162970,0.150449,0.142837,0.139963,...,0.351219,0.360685,0.352973,0.343354,0.329605,0.316874,0.296893,0.274650,0.241907,0.207499
2012-04,0.152400,0.217229,0.197744,0.175748,0.156256,0.142692,0.134073,0.128980,0.127899,0.132474,...,0.307767,0.318279,0.320971,0.311462,0.296848,0.282221,0.252988,0.219906,0.188952,0.164874
2012-05,0.134830,0.154574,0.140291,0.126945,0.116503,0.110832,0.107758,0.105116,0.105975,0.111560,...,0.254587,0.257347,0.262779,0.268836,0.265439,0.255529,0.230154,0.200542,0.172461,0.149573
2012-06,0.132968,0.135052,0.122561,0.111412,0.104188,0.101452,0.099064,0.097811,0.099015,0.103042,...,0.245224,0.245720,0.246870,0.248621,0.249927,0.245282,0.224558,0.198418,0.172302,0.149490
2012-07,0.131606,0.130598,0.118536,0.109373,0.104069,0.100448,0.097798,0.096398,0.097152,0.100576,...,0.234815,0.236610,0.239133,0.245179,0.245399,0.238080,0.217509,0.192570,0.167802,0.146468


In [66]:
df_year_quarterly_mean

,hh_0,hh_1,hh_2,hh_3,hh_4,hh_5,hh_6,hh_7,hh_8,hh_9,...,hh_38,hh_39,hh_40,hh_41,hh_42,hh_43,hh_44,hh_45,hh_46,hh_47
day,,,,,,,,,,,,,,,,,,,,,
Q1,0.235165,0.217951,0.193398,0.173360,0.157738,0.147590,0.140371,0.135064,0.131382,0.130727,...,0.396800,0.395455,0.384248,0.372587,0.360016,0.345776,0.326322,0.305572,0.273648,0.238006
Q2,0.148033,0.144582,0.131403,0.120476,0.113115,0.109181,0.106194,0.104876,0.105734,0.111035,...,0.262895,0.267317,0.270509,0.272066,0.266687,0.256305,0.231832,0.202831,0.175343,0.152176
Q3,0.135707,0.128495,0.117184,0.108974,0.104078,0.101231,0.098623,0.097413,0.098128,0.102239,...,0.247713,0.251273,0.252002,0.251085,0.244040,0.233942,0.212482,0.187738,0.163967,0.144075
Q4,0.194900,0.183465,0.163067,0.146724,0.134349,0.126620,0.121055,0.117768,0.116080,0.117794,...,0.367025,0.360438,0.349134,0.337979,0.324647,0.310471,0.290198,0.267588,0.239146,0.209359


In [67]:
df_year_monthly_mean

,hh_0,hh_1,hh_2,hh_3,hh_4,hh_5,hh_6,hh_7,hh_8,hh_9,...,hh_38,hh_39,hh_40,hh_41,hh_42,hh_43,hh_44,hh_45,hh_46,hh_47
day,,,,,,,,,,,,,,,,,,,,,
M1,0.237189,0.213976,0.189298,0.169251,0.154328,0.144411,0.137603,0.132314,0.128726,0.127993,...,0.402798,0.399526,0.387023,0.375134,0.362133,0.347177,0.327825,0.307900,0.275760,0.239747
M10,0.147806,0.143043,0.129777,0.119244,0.111496,0.106990,0.104197,0.103379,0.104203,0.109397,...,0.332048,0.322271,0.310346,0.298499,0.282741,0.266386,0.240255,0.211959,0.183747,0.159438
M11,0.205350,0.193436,0.170217,0.152056,0.138186,0.129542,0.123537,0.119385,0.116812,0.117402,...,0.378352,0.374059,0.363058,0.352066,0.339470,0.325900,0.307865,0.287524,0.258334,0.225242
M12,0.230379,0.212962,0.188373,0.168158,0.152744,0.142778,0.134956,0.130114,0.126845,0.126274,...,0.389970,0.384273,0.373278,0.362636,0.350942,0.338291,0.321535,0.302247,0.274299,0.242378
M2,0.233749,0.216688,0.192232,0.172659,0.157256,0.147310,0.140330,0.135290,0.131544,0.131055,...,0.395093,0.393565,0.382803,0.370851,0.358068,0.344152,0.324493,0.303561,0.272260,0.236685
M3,0.233903,0.226678,0.202186,0.181445,0.164294,0.153439,0.145153,0.139403,0.135658,0.134878,...,0.389219,0.391439,0.381753,0.370930,0.359420,0.345897,0.326587,0.304715,0.272197,0.237083
M4,0.167618,0.169459,0.155181,0.142036,0.131871,0.125439,0.121091,0.119176,0.118981,0.124338,...,0.302997,0.314523,0.317412,0.310667,0.296189,0.279501,0.251026,0.218938,0.188960,0.163826
M5,0.143250,0.140137,0.126910,0.116299,0.109459,0.105811,0.103411,0.102149,0.103429,0.109290,...,0.256222,0.259179,0.264506,0.269496,0.264071,0.252364,0.227619,0.198866,0.171242,0.148405
M6,0.137621,0.129853,0.117551,0.108011,0.102290,0.099985,0.097486,0.096567,0.097832,0.102578,...,0.238697,0.239093,0.240546,0.245221,0.246771,0.242384,0.221191,0.194296,0.168817,0.146837


In [68]:
df_dayofweek_mean

,hh_0,hh_1,hh_2,hh_3,hh_4,hh_5,hh_6,hh_7,hh_8,hh_9,...,hh_38,hh_39,hh_40,hh_41,hh_42,hh_43,hh_44,hh_45,hh_46,hh_47
day,,,,,,,,,,,,,,,,,,,,,
0,0.175689,0.165756,0.148915,0.135408,0.125732,0.119690,0.115129,0.112522,0.111899,0.114931,...,0.328055,0.328702,0.325068,0.318206,0.308557,0.294844,0.270457,0.243551,0.212463,0.183091
1,0.174443,0.164973,0.148070,0.135002,0.125382,0.119641,0.115421,0.112848,0.112525,0.115719,...,0.323757,0.325141,0.319816,0.315689,0.305162,0.292160,0.268636,0.242715,0.212326,0.183450
2,0.175248,0.165085,0.148343,0.135188,0.125460,0.119637,0.115592,0.113150,0.112579,0.115691,...,0.320613,0.322116,0.318184,0.313606,0.303705,0.290953,0.268544,0.243409,0.213778,0.185286
3,0.176654,0.166573,0.149173,0.135665,0.125957,0.120320,0.115982,0.113386,0.112765,0.116054,...,0.317018,0.318254,0.314734,0.310774,0.301600,0.289201,0.267418,0.243101,0.214596,0.186872
4,0.178309,0.167695,0.149891,0.136132,0.126346,0.120517,0.116123,0.113633,0.112878,0.116311,...,0.313438,0.311456,0.306077,0.299135,0.290217,0.279377,0.260936,0.239813,0.215558,0.191198
5,0.184965,0.174826,0.156625,0.141648,0.130883,0.124014,0.118795,0.115613,0.113955,0.114982,...,0.312575,0.308813,0.302039,0.294194,0.285438,0.274266,0.258864,0.239746,0.217597,0.194776
6,0.189022,0.179145,0.160622,0.144918,0.133259,0.125604,0.119877,0.116062,0.113600,0.114207,...,0.330060,0.328870,0.322138,0.315209,0.303493,0.291185,0.268226,0.242262,0.213309,0.184548


In [69]:
def plot_with_seasonal_colors(df, title_suffix):
    plt.figure(figsize=(14, 8))

    # 定义季节或月份颜色映射
    # 直接通过字符串切片提取出所有唯一的季节/月份
    unique_seasons = sorted({str(index)[-2:] for index in df.index})
    season_colors = plt.cm.viridis(np.linspace(0, 1, len(unique_seasons)))
    color_map = dict(zip(unique_seasons, season_colors))

    # x轴标签
    x_labels = [f'{i // 2:02d}:{i % 2 * 30:02d}' for i in range(48)]

    # 遍历DataFrame的每一行
    for index, row in df.iterrows():
        year, season = str(index)[:-2], str(index)[-2:]
        # 确保row.values中的所有值都是可以绘图的数值类型
        values = [float(value) for value in row.values if isinstance(value, (int, float))]

        # 为这个季节/月份的行选择颜色
        color = color_map[season]

        # 绘制每半小时的平均用电量
        plt.plot(x_labels, values, marker='o', linestyle='-', label=f'{year} {season}', color=color)

    # 自定义图例和图表布局
    plt.legend(title="Time Period", loc='upper right', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.title(f'Average Daily Electricity Consumption per 30 Minutes - {title_suffix}')
    plt.xlabel('Time of Day (in 30-minute intervals)')
    plt.ylabel('Average Electricity Consumption (kWh)')
    plt.xticks(range(0, 48, 3), labels=x_labels[::3])
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [70]:
plot_level(1, lambda: plot_with_seasonal_colors(df_yearly_mean, 'Yearly'))

In [71]:
plot_level(1, lambda: plot_with_seasonal_colors(df_quarterly_mean, 'Quarterly'))

In [72]:
plot_level(1, lambda: plot_with_seasonal_colors(df_monthly_mean, 'Monthly'))

In [73]:
plot_level(1, lambda: plot_with_seasonal_colors(df_year_quarterly_mean, 'Yearly Quarterly'))

In [74]:
plot_level(1, lambda: plot_with_seasonal_colors(df_year_monthly_mean, 'Yearly Monthly'))

In [75]:
plot_level(1, lambda: plot_with_seasonal_colors(df_dayofweek_mean, 'Day of Week'))

## uk_bank_holidays

### 数据查看与清洗

In [76]:
uk_bank_holidays

,Bank holidays,Type
0,2012-12-26,Boxing Day
1,2012-12-25,Christmas Day
2,2012-08-27,Summer bank holiday
3,2012-05-06,Queen?s Diamond Jubilee (extra bank holiday)
4,2012-04-06,Spring bank holiday (substitute day)
5,2012-07-05,Early May bank holiday
6,2012-09-04,Easter Monday
7,2012-06-04,Good Friday
8,2012-02-01,New Year?s Day (substitute day)
9,2013-12-26,Boxing Day


In [77]:
def process_uk_bank_holidays(df_bank_holidays, df_hhblock):
    # 将'Bank holidays'列转换为日期时间格式
    df_bank_holidays['Bank holidays'] = pd.to_datetime(df_bank_holidays['Bank holidays'])

    # 首先将uk_bank_holidays的日期列转换为日期格式
    df_bank_holidays['Bank holidays'] = pd.to_datetime(df_bank_holidays['Bank holidays'])
    # 创建一个集合，包含所有的假期日期
    holidays_set = set(df_bank_holidays['Bank holidays'])

    # 使用.copy()确保filtered_big_hhblock是一个独立的副本，避免后续修改影响原始数据
    filtered_big_hhblock_copy = df_hhblock.copy()
    
    filtered_big_hhblock_copy.loc[:, 'is_holiday'] = filtered_big_hhblock_copy['day'].isin(holidays_set)
    
    df_numeric = filtered_big_hhblock_copy.iloc[:, 2:]  # 选择数值列
    
    # 根据is_holiday列分组求均值
    df_holidays_mean = df_numeric.groupby(filtered_big_hhblock_copy['is_holiday']).mean()
    
    # 重命名索引以更清晰地表示数据
    df_holidays_mean.index = ['out_holidays', 'in_holidays']

    return df_holidays_mean

In [78]:
%%time
df_holidays_mean = process_uk_bank_holidays(uk_bank_holidays, filtered_big_hhblock)

CPU times: total: 141 ms
Wall time: 850 ms


In [79]:
def plot_average_daily_consumption(df, title_suffix):
    plt.figure(figsize=(14, 8))

    # 创建表示24小时内每半小时间隔的x轴标签列表
    x_labels = [f'{i//2:02d}:{i%2*30:02d}' for i in range(48)]

    for index, row in df.iterrows():
        # 转换index为字符串，适用于任何形式的Period或其他索引类型
        index_str = str(index)

        # 确保row.values中的所有值都是可以绘图的数值类型
        # 这里直接使用row.values[:-1]来排除可能的非数值列
        # 假设最后一列是非数值列，如果不是，请根据实际情况调整
        values = [float(value) for value in row.values if isinstance(value, (int, float))]

        # 绘制每半小时的平均用电量
        plt.plot(x_labels, values, marker='o', linestyle='-', label=index_str)

    plt.legend(title="Time Period", loc='upper right')
    plt.title(f'Average Daily Electricity Consumption per 30 Minutes - {title_suffix}')
    plt.xlabel('Time of Day (in 30-minute intervals)')
    plt.ylabel('Average Electricity Consumption (kWh)')
    plt.xticks(range(0, 48, 3), labels=x_labels[::3])
    plt.grid(True)
    plt.show()

In [80]:
plot_level(1, lambda: plot_average_daily_consumption(df_holidays_mean.iloc[:, :-1], 'Holidays'))

## Acorn数据分析与可视化

In [81]:
# 合并DataFrame
merged_info_hhblock_df = pd.merge(filtered_big_hhblock,
                                  informations_households_filtered[['LCLid', 'stdorToU', 'Acorn', 'Acorn_grouped']],
                                  on='LCLid', how='left')

In [82]:
merged_info_hhblock_df.head(1000)

,LCLid,day,hh_0,hh_1,hh_2,hh_3,hh_4,hh_5,hh_6,hh_7,...,hh_41,hh_42,hh_43,hh_44,hh_45,hh_46,hh_47,stdorToU,Acorn,Acorn_grouped
0,MAC000002,2012-10-13,0.263,0.269,0.275,0.256,0.211,0.136,0.161,0.119,...,0.239,0.230,0.233,0.235,0.188,0.259,0.250,Std,ACORN-A,Affluent
1,MAC000002,2012-10-14,0.262,0.166,0.226,0.088,0.126,0.082,0.123,0.083,...,0.745,0.712,0.511,0.231,0.210,0.278,0.159,Std,ACORN-A,Affluent
2,MAC000002,2012-10-15,0.192,0.097,0.141,0.083,0.132,0.070,0.130,0.074,...,0.258,0.260,0.334,0.299,0.236,0.241,0.237,Std,ACORN-A,Affluent
3,MAC000002,2012-10-16,0.237,0.237,0.193,0.118,0.098,0.107,0.094,0.109,...,0.228,0.203,0.211,0.188,0.213,0.157,0.202,Std,ACORN-A,Affluent
4,MAC000002,2012-10-17,0.157,0.211,0.155,0.169,0.101,0.117,0.084,0.118,...,0.208,0.265,0.377,0.327,0.277,0.288,0.256,Std,ACORN-A,Affluent
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,MAC000246,2013-04-15,0.702,0.764,0.356,0.041,0.055,0.050,0.039,0.093,...,1.241,1.039,1.209,0.860,0.457,0.802,0.742,Std,ACORN-A,Affluent
996,MAC000246,2013-04-16,0.640,0.095,0.748,0.693,0.302,0.024,0.029,0.072,...,2.186,2.239,1.239,1.003,0.685,0.887,0.550,Std,ACORN-A,Affluent
997,MAC000246,2013-04-17,0.600,0.776,0.453,0.064,0.057,0.045,0.058,0.024,...,1.081,0.353,0.731,0.836,0.701,0.233,0.171,Std,ACORN-A,Affluent
998,MAC000246,2013-04-18,0.134,0.120,0.277,0.235,0.148,0.749,0.504,0.073,...,2.145,1.430,1.486,1.099,0.960,0.761,0.215,Std,ACORN-A,Affluent


In [83]:
def data_for_acorn_plot(merged_info_hhblock_df):
    # 选择所有数值列用于分组聚合
    df_numeric = merged_info_hhblock_df.iloc[:, 2:-3]  # 选择从第三列到最后的所有列
    
    # 按'stdorToU'分组求均值
    df_std_tou_mean = df_numeric.groupby(merged_info_hhblock_df['stdorToU']).mean()
    
    # 按'Acorn'分组求均值
    df_acorn_mean = df_numeric.groupby(merged_info_hhblock_df['Acorn']).mean()
    
    # 按'Acorn_grouped'分组求均值
    df_acorn_grouped_mean = df_numeric.groupby(merged_info_hhblock_df['Acorn_grouped']).mean()
    
    return df_std_tou_mean, df_acorn_mean, df_acorn_grouped_mean

In [84]:
%%time
df_std_tou_mean, df_acorn_mean, df_acorn_grouped_mean = data_for_acorn_plot(merged_info_hhblock_df)

CPU times: total: 219 ms
Wall time: 1.4 s


In [85]:
df_std_tou_mean

,hh_0,hh_1,hh_2,hh_3,hh_4,hh_5,hh_6,hh_7,hh_8,hh_9,...,hh_38,hh_39,hh_40,hh_41,hh_42,hh_43,hh_44,hh_45,hh_46,hh_47
stdorToU,,,,,,,,,,,,,,,,,,,,,
Std,0.185532,0.177643,0.159185,0.144048,0.132801,0.125803,0.120766,0.117434,0.116304,0.118851,...,0.323984,0.324053,0.319433,0.313533,0.303504,0.290602,0.268844,0.244293,0.216493,0.189280
ToU,0.153965,0.135380,0.121753,0.112506,0.106794,0.103628,0.100550,0.099791,0.099298,0.101755,...,0.308110,0.306311,0.299600,0.293750,0.284830,0.274854,0.255495,0.233325,0.205239,0.178074


In [86]:
df_acorn_mean

,hh_0,hh_1,hh_2,hh_3,hh_4,hh_5,hh_6,hh_7,hh_8,hh_9,...,hh_38,hh_39,hh_40,hh_41,hh_42,hh_43,hh_44,hh_45,hh_46,hh_47
Acorn,,,,,,,,,,,,,,,,,,,,,
ACORN-A,0.310594,0.275233,0.249999,0.233997,0.223660,0.217964,0.214212,0.212482,0.213791,0.217612,...,0.621023,0.618901,0.613864,0.607231,0.594865,0.567839,0.527554,0.475788,0.416736,0.359784
ACORN-B,0.176595,0.153550,0.135713,0.127037,0.122915,0.123745,0.123675,0.124608,0.124809,0.126982,...,0.406199,0.401831,0.385444,0.373999,0.367419,0.350518,0.324503,0.288066,0.247814,0.208473
ACORN-C,0.180511,0.156411,0.140311,0.129169,0.121903,0.117881,0.114905,0.114885,0.113887,0.117030,...,0.401863,0.393627,0.381439,0.373072,0.359463,0.343767,0.313998,0.284920,0.245768,0.210174
ACORN-D,0.215055,0.188244,0.168507,0.156437,0.147990,0.143163,0.139424,0.138753,0.138603,0.140139,...,0.457504,0.463231,0.458638,0.448370,0.433686,0.416672,0.384734,0.343719,0.299946,0.253789
ACORN-E,0.194509,0.188161,0.169465,0.153901,0.142367,0.135577,0.130344,0.125761,0.124095,0.126568,...,0.324501,0.327997,0.323897,0.317185,0.306325,0.293762,0.272571,0.248094,0.220153,0.191844
ACORN-F,0.156744,0.140010,0.128291,0.118938,0.111147,0.105000,0.100997,0.099255,0.098745,0.101192,...,0.286275,0.288135,0.285672,0.282226,0.274664,0.263536,0.246060,0.225132,0.202099,0.178210
ACORN-G,0.165387,0.150853,0.135291,0.125210,0.117683,0.112944,0.116979,0.116311,0.108730,0.115878,...,0.335059,0.325954,0.316183,0.308423,0.299972,0.286508,0.264702,0.242130,0.212200,0.184461
ACORN-H,0.171134,0.151252,0.135965,0.123986,0.117435,0.114038,0.110737,0.110551,0.111230,0.114085,...,0.366944,0.360537,0.352982,0.346635,0.335639,0.322592,0.295274,0.266071,0.230752,0.198722
ACORN-I,0.137267,0.121608,0.111217,0.104484,0.101162,0.099284,0.097319,0.097159,0.095106,0.096003,...,0.299042,0.296769,0.291644,0.289498,0.281951,0.269198,0.247433,0.219366,0.187023,0.159960


In [87]:
df_acorn_grouped_mean

,hh_0,hh_1,hh_2,hh_3,hh_4,hh_5,hh_6,hh_7,hh_8,hh_9,...,hh_38,hh_39,hh_40,hh_41,hh_42,hh_43,hh_44,hh_45,hh_46,hh_47
Acorn_grouped,,,,,,,,,,,,,,,,,,,,,
Adversity,0.155153,0.149892,0.134126,0.120670,0.110920,0.104401,0.098853,0.095930,0.095351,0.097542,...,0.263342,0.261277,0.256085,0.251144,0.242884,0.232900,0.215803,0.198104,0.177161,0.157519
Affluent,0.203860,0.191560,0.172465,0.157691,0.146995,0.140794,0.136055,0.132530,0.131325,0.133800,...,0.367639,0.370243,0.365378,0.358041,0.346439,0.332082,0.307599,0.278687,0.245735,0.212568
Comfortable,0.171179,0.158756,0.141604,0.128282,0.118529,0.112612,0.109210,0.107593,0.106387,0.109404,...,0.319896,0.317342,0.312220,0.307358,0.298380,0.286318,0.264808,0.240297,0.211733,0.184342


In [88]:
plot_level(1, lambda: plot_average_daily_consumption(df_std_tou_mean, 'Std or ToU'))

In [89]:
plot_level(1, lambda: plot_with_seasonal_colors(df_acorn_mean, 'Acorn'))

In [90]:
plot_level(1, lambda: plot_average_daily_consumption(df_acorn_grouped_mean, 'Acorn Grouped'))

(不同富裕程度的周）

## Weather Data

temperatureMax - 当天的最高气温。
temperatureMaxTime - 最高气温发生的时间。
windBearing - 风向，表示风从哪个方向吹来，通常以角度表示。
icon - 天气情况的图标代码，如晴天、多云等。
dewPoint - 露点温度，表示空气达到饱和（露水开始凝结）的温度。
temperatureMinTime - 最低气温发生的时间。
cloudCover - 云量，表示天空被云层覆盖的比例。
windSpeed - 风速。
pressure - 大气压强。
apparentTemperatureMinTime - 体感最低温度发生的时间。
apparentTemperatureHigh - 当天的最高体感温度。
precipType - 降水类型，如雨、雪。
visibility - 能见度。
humidity - 湿度。
apparentTemperatureHighTime - 最高体感温度发生的时间。
apparentTemperatureLow - 当天的最低体感温度。
apparentTemperatureMax - 当天的最高体感温度。
uvIndex - 紫外线指数。
time - 观测时间。
sunsetTime - 日落时间。
temperatureLow - 当天的最低气温。
temperatureMin - 同temperatureLow，当天的最低气温。
temperatureHigh - 当天的最高气温。
sunriseTime - 日出时间。
temperatureHighTime - 最高气温发生的时间。
uvIndexTime - 紫外线指数达到最高点的时间。
summary - 天气概况的文字描述。
temperatureLowTime - 最低气温发生的时间。
apparentTemperatureMin - 当天的最低体感温度。
apparentTemperatureMaxTime - 最高体感温度发生的时间。
apparentTemperatureLowTime - 最低体感温度发生的时间。
moonPhase - 月相。

In [91]:
weather_daily_darksky

,temperatureMax,temperatureMaxTime,windBearing,icon,dewPoint,temperatureMinTime,cloudCover,windSpeed,pressure,apparentTemperatureMinTime,...,temperatureHigh,sunriseTime,temperatureHighTime,uvIndexTime,summary,temperatureLowTime,apparentTemperatureMin,apparentTemperatureMaxTime,apparentTemperatureLowTime,moonPhase
0,11.96,2011-11-11 23:00:00,123,fog,9.40,2011-11-11 07:00:00,0.79,3.88,1016.08,2011-11-11 07:00:00,...,10.87,2011-11-11 07:12:14,2011-11-11 19:00:00,2011-11-11 11:00:00,Foggy until afternoon.,2011-11-11 19:00:00,6.48,2011-11-11 23:00:00,2011-11-11 19:00:00,0.52
1,8.59,2011-12-11 14:00:00,198,partly-cloudy-day,4.49,2011-12-11 01:00:00,0.56,3.94,1007.71,2011-12-11 02:00:00,...,8.59,2011-12-11 07:57:02,2011-12-11 14:00:00,2011-12-11 12:00:00,Partly cloudy throughout the day.,2011-12-12 07:00:00,0.11,2011-12-11 20:00:00,2011-12-12 08:00:00,0.53
2,10.33,2011-12-27 02:00:00,225,partly-cloudy-day,5.47,2011-12-27 23:00:00,0.85,3.54,1032.76,2011-12-27 22:00:00,...,10.33,2011-12-27 08:07:06,2011-12-27 14:00:00,2011-12-27 00:00:00,Mostly cloudy throughout the day.,2011-12-27 23:00:00,5.59,2011-12-27 02:00:00,2011-12-28 00:00:00,0.10
3,8.07,2011-12-02 23:00:00,232,wind,3.69,2011-12-02 07:00:00,0.32,3.00,1012.12,2011-12-02 07:00:00,...,7.36,2011-12-02 07:46:09,2011-12-02 12:00:00,2011-12-02 10:00:00,Partly cloudy throughout the day and breezy ov...,2011-12-02 19:00:00,0.46,2011-12-02 12:00:00,2011-12-02 19:00:00,0.25
4,8.22,2011-12-24 23:00:00,252,partly-cloudy-night,2.79,2011-12-24 07:00:00,0.37,4.46,1028.17,2011-12-24 07:00:00,...,7.93,2011-12-24 08:06:15,2011-12-24 15:00:00,2011-12-24 13:00:00,Mostly cloudy throughout the day.,2011-12-24 19:00:00,-0.51,2011-12-24 23:00:00,2011-12-24 20:00:00,0.99
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
877,9.03,2014-01-26 16:00:00,233,partly-cloudy-day,2.39,2014-01-26 21:00:00,0.40,4.55,1002.10,2014-01-26 22:00:00,...,9.03,2014-01-26 07:48:49,2014-01-26 16:00:00,2014-01-26 11:00:00,Mostly cloudy until evening.,2014-01-27 05:00:00,-1.30,2014-01-26 15:00:00,2014-01-27 04:00:00,0.84
878,10.31,2014-02-27 14:00:00,224,partly-cloudy-day,3.08,2014-02-27 23:00:00,0.32,4.14,1007.02,2014-02-27 22:00:00,...,10.31,2014-02-27 06:51:45,2014-02-27 14:00:00,2014-02-27 12:00:00,Partly cloudy until evening.,2014-02-28 02:00:00,1.41,2014-02-27 14:00:00,2014-02-28 02:00:00,0.93
879,18.97,2014-03-09 14:00:00,172,partly-cloudy-night,4.30,2014-03-09 07:00:00,0.04,2.78,1022.44,2014-03-09 07:00:00,...,18.97,2014-03-09 06:29:49,2014-03-09 14:00:00,2014-03-09 12:00:00,Partly cloudy in the evening.,2014-03-10 05:00:00,7.08,2014-03-09 14:00:00,2014-03-10 06:00:00,0.28
880,8.83,2014-02-12 16:00:00,210,wind,1.94,2014-02-12 01:00:00,0.59,7.24,994.27,2014-02-12 01:00:00,...,8.83,2014-02-12 07:21:44,2014-02-12 16:00:00,2014-02-12 10:00:00,Mostly cloudy until evening and breezy through...,2014-02-13 05:00:00,-1.20,2014-02-12 16:00:00,2014-02-13 02:00:00,0.42


In [92]:
weather_daily_darksky.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 882 entries, 0 to 881
Data columns (total 32 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   temperatureMax               882 non-null    float64
 1   temperatureMaxTime           882 non-null    object 
 2   windBearing                  882 non-null    int64  
 3   icon                         882 non-null    object 
 4   dewPoint                     882 non-null    float64
 5   temperatureMinTime           882 non-null    object 
 6   cloudCover                   881 non-null    float64
 7   windSpeed                    882 non-null    float64
 8   pressure                     882 non-null    float64
 9   apparentTemperatureMinTime   882 non-null    object 
 10  apparentTemperatureHigh      882 non-null    float64
 11  precipType                   882 non-null    object 
 12  visibility                   882 non-null    float64
 13  humidity            

In [93]:
missing_weather_daily_darksky_values = weather_daily_darksky.isnull().sum()
missing_weather_daily_darksky_values

temperatureMax                 0
temperatureMaxTime             0
windBearing                    0
icon                           0
dewPoint                       0
temperatureMinTime             0
cloudCover                     1
windSpeed                      0
pressure                       0
apparentTemperatureMinTime     0
apparentTemperatureHigh        0
precipType                     0
visibility                     0
humidity                       0
apparentTemperatureHighTime    0
apparentTemperatureLow         0
apparentTemperatureMax         0
uvIndex                        1
time                           0
sunsetTime                     0
temperatureLow                 0
temperatureMin                 0
temperatureHigh                0
sunriseTime                    0
temperatureHighTime            0
uvIndexTime                    1
summary                        0
temperatureLowTime             0
apparentTemperatureMin         0
apparentTemperatureMaxTime     0
apparentTe

visibility - 能见度，通常以公里或英里为单位，表示在特定天气条件下最远可见的距离。
windBearing - 风向，表示风从哪个方向吹来。通常以角度表示，其中0度代表北风，90度代表东风，180度代表南风，270度代表西风。
temperature - 气温，通常以摄氏度或华氏度为单位，表示空气的热度。
time - 时间，数据采集的具体时间点，通常以UNIX时间戳或可读格式表示。
dewPoint - 露点温度，当空气冷却到无法容纳其中所有水蒸气时，水蒸气凝结成露水的温度。
pressure - 大气压强，表示空气的重量压在地面上的力量，通常以百帕斯卡（hPa）或毫巴（mb）为单位。
apparentTemperature - 体感温度，综合考虑风速、湿度和实际气温对人体感知温度的影响。
windSpeed - 风速，表示风的快慢，通常以每秒米数或每小时英里数表示。
precipType - 降水类型，如雨、雪、冰雹等。
icon - 天气图标的标识，用于直观表示天气状况，如晴天、多云、雨天等。
humidity - 湿度，表示空气中水蒸气的含量，通常以百分比表示。
summary - 天气概况的文字描述，提供对当前或预测天气状况的简短总结。

In [94]:
weather_hourly_darksky.head(1000)

,visibility,windBearing,temperature,time,dewPoint,pressure,apparentTemperature,windSpeed,precipType,icon,humidity,summary
0,5.97,104,10.24,2011-11-11 00:00:00,8.86,1016.76,10.24,2.77,rain,partly-cloudy-night,0.91,Partly Cloudy
1,4.88,99,9.76,2011-11-11 01:00:00,8.83,1016.63,8.24,2.95,rain,partly-cloudy-night,0.94,Partly Cloudy
2,3.70,98,9.46,2011-11-11 02:00:00,8.79,1016.36,7.76,3.17,rain,partly-cloudy-night,0.96,Partly Cloudy
3,3.12,99,9.23,2011-11-11 03:00:00,8.63,1016.28,7.44,3.25,rain,fog,0.96,Foggy
4,1.85,111,9.26,2011-11-11 04:00:00,9.21,1015.98,7.24,3.70,rain,fog,1.00,Foggy
...,...,...,...,...,...,...,...,...,...,...,...,...
995,13.07,252,11.63,2011-12-31 11:00:00,9.73,1010.91,11.63,5.24,rain,partly-cloudy-day,0.88,Mostly Cloudy
996,13.52,251,11.88,2011-12-31 12:00:00,9.77,1010.65,11.88,6.08,rain,partly-cloudy-day,0.87,Mostly Cloudy
997,13.07,252,12.16,2011-12-31 13:00:00,10.09,1010.47,12.16,6.00,rain,partly-cloudy-day,0.87,Mostly Cloudy
998,12.89,251,12.17,2011-12-31 14:00:00,9.74,1010.15,12.17,5.98,rain,partly-cloudy-day,0.85,Mostly Cloudy


In [95]:
# 确保time列为datetime格式
weather_hourly_darksky['time'] = pd.to_datetime(weather_hourly_darksky['time'])

In [96]:
weather_hourly_darksky.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21165 entries, 0 to 21164
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   visibility           21165 non-null  float64       
 1   windBearing          21165 non-null  int64         
 2   temperature          21165 non-null  float64       
 3   time                 21165 non-null  datetime64[ns]
 4   dewPoint             21165 non-null  float64       
 5   pressure             21152 non-null  float64       
 6   apparentTemperature  21165 non-null  float64       
 7   windSpeed            21165 non-null  float64       
 8   precipType           21165 non-null  object        
 9   icon                 21165 non-null  object        
 10  humidity             21165 non-null  float64       
 11  summary              21165 non-null  object        
dtypes: datetime64[ns](1), float64(7), int64(1), object(3)
memory usage: 1.9+ MB


### 缺失值处理

In [97]:
missing_weather_hourly_darksky_values = weather_hourly_darksky.isnull().sum()
missing_weather_hourly_darksky_values

visibility              0
windBearing             0
temperature             0
time                    0
dewPoint                0
pressure               13
apparentTemperature     0
windSpeed               0
precipType              0
icon                    0
humidity                0
summary                 0
dtype: int64

In [98]:
weather_hourly_darksky['pressure'] = weather_hourly_darksky['pressure'].ffill()

In [99]:
weather_hourly_darksky.info()
missing_weather_hourly_darksky_values = weather_hourly_darksky.isnull().sum()
missing_weather_hourly_darksky_values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21165 entries, 0 to 21164
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   visibility           21165 non-null  float64       
 1   windBearing          21165 non-null  int64         
 2   temperature          21165 non-null  float64       
 3   time                 21165 non-null  datetime64[ns]
 4   dewPoint             21165 non-null  float64       
 5   pressure             21165 non-null  float64       
 6   apparentTemperature  21165 non-null  float64       
 7   windSpeed            21165 non-null  float64       
 8   precipType           21165 non-null  object        
 9   icon                 21165 non-null  object        
 10  humidity             21165 non-null  float64       
 11  summary              21165 non-null  object        
dtypes: datetime64[ns](1), float64(7), int64(1), object(3)
memory usage: 1.9+ MB


visibility             0
windBearing            0
temperature            0
time                   0
dewPoint               0
pressure               0
apparentTemperature    0
windSpeed              0
precipType             0
icon                   0
humidity               0
summary                0
dtype: int64

In [100]:
def plot_weather_boxplot(df, column_name, time_type):
    # 确保time列为datetime格式
    if not pd.api.types.is_datetime64_any_dtype(df['time']):
        df['time'] = pd.to_datetime(df['time'])

    # 根据时间类型提取对应的时间单位
    if time_type == 'hour':
        df['time_unit'] = df['time'].dt.hour
    elif time_type == 'day':
        df['time_unit'] = df['time'].dt.date
    elif time_type == 'month':
        df['time_unit'] = df['time'].dt.month
    elif time_type == 'quarter':
        df['time_unit'] = df['time'].dt.quarter
    elif time_type == 'year':
        df['time_unit'] = df['time'].dt.year
    else:
        raise ValueError("Invalid time_type provided. Choose from 'hour', 'day', 'month', 'quarter', 'year'.")

    # 绘制箱线图
    plt.figure(figsize=(12, 8))
    sns.boxplot(x='time_unit', y=column_name, data=df)
    plt.title(f'Box plot of {column_name} over {time_type}')
    plt.xlabel(time_type.capitalize())
    plt.ylabel(column_name.capitalize())
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.show()


In [101]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'temperature', 'hour'))

In [102]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'apparentTemperature', 'hour'))

In [103]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'temperature', 'day'))

In [104]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'apparentTemperature', 'day'))

In [105]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'temperature', 'month'))

In [106]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'apparentTemperature', 'month'))

In [107]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'temperature', 'quarter'))

In [108]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'apparentTemperature', 'quarter'))

In [109]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'temperature', 'year'))

In [110]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'apparentTemperature', 'year'))

In [111]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'visibility', 'hour'))

In [112]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'windBearing', 'month'))

In [113]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'dewPoint', 'month'))

In [114]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'pressure', 'month'))

In [115]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'humidity', 'hour'))

In [116]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'humidity', 'month'))

In [117]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'windSpeed', 'hour'))

In [118]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'windSpeed', 'quarter'))

In [119]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'precipType', 'month'))

In [120]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'icon', 'month'))

In [121]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'icon', 'quarter'))

In [122]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'summary', 'month'))

In [123]:
def plot_weather_scatterplot(df, column_name1, column_name2):
    # 绘制散点图
    plt.figure(figsize=(12, 8))
    sns.scatterplot(x=column_name1, y=column_name2, data=df)
    plt.title(f'Scatter plot of {column_name1} vs. {column_name2}')
    plt.xlabel(column_name1.capitalize())
    plt.ylabel(column_name2.capitalize())
    plt.grid(True)
    plt.show()

In [124]:
plot_level(1, lambda: plot_weather_scatterplot(weather_hourly_darksky, 'temperature', 'apparentTemperature'))

In [125]:
def plot_weather_lineplot(df, column_name, time_type):
    # 确保time列为datetime格式
    if not pd.api.types.is_datetime64_any_dtype(df['time']):
        df['time'] = pd.to_datetime(df['time'])

    # 根据时间类型提取对应的时间单位
    if time_type == 'hour':
        df['time_unit'] = df['time'].dt.hour
    elif time_type == 'day':
        df['time_unit'] = df['time'].dt.date
    elif time_type == 'month':
        df['time_unit'] = df['time'].dt.month
    elif time_type == 'quarter':
        df['time_unit'] = df['time'].dt.quarter
    elif time_type == 'year':
        df['time_unit'] = df['time'].dt.year
    else:
        raise ValueError("Invalid time_type provided. Choose from 'day', 'month', 'quarter', 'year'.")

    # 绘制折线图
    plt.figure(figsize=(12, 8))
    sns.lineplot(x='time_unit', y=column_name, data=df)
    plt.title(f'Line plot of {column_name} over {time_type}')
    plt.xlabel(time_type.capitalize())
    plt.ylabel(column_name.capitalize())
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.show()

In [126]:
plot_level(2, lambda: plot_weather_lineplot(weather_hourly_darksky, 'temperature', 'hour'))

In [127]:
plot_level(2, lambda: plot_weather_lineplot(weather_hourly_darksky, 'apparentTemperature', 'hour'))

In [128]:
plot_level(2, lambda: plot_weather_lineplot(weather_hourly_darksky, 'temperature', 'day'))

In [129]:
plot_level(2, lambda: plot_weather_lineplot(weather_hourly_darksky, 'apparentTemperature', 'day'))

In [130]:
plot_level(2, lambda: plot_weather_lineplot(weather_hourly_darksky, 'temperature', 'month'))

In [131]:
plot_level(2, lambda: plot_weather_lineplot(weather_hourly_darksky, 'apparentTemperature', 'month'))

In [132]:
plot_level(2, lambda: plot_weather_lineplot(weather_hourly_darksky, 'temperature', 'quarter'))

In [133]:
plot_level(2, lambda: plot_weather_lineplot(weather_hourly_darksky, 'apparentTemperature', 'quarter'))

In [134]:
plot_level(2, lambda: plot_weather_lineplot(weather_hourly_darksky, 'temperature', 'year'))

In [135]:
plot_level(2, lambda: plot_weather_lineplot(weather_hourly_darksky, 'apparentTemperature', 'year'))

# 时间序列聚类分析
1. 数据预处理
1.1 同步数据时间粒度
家庭用电量数据：以半小时为单位，需要将其转换为小时单位以匹配天气数据的时间粒度。这可以通过取每小时内两个半小时用电量的平均值来实现。
天气数据：已经以小时为单位，直接使用。
1.2 时间对齐
确保两个数据集的时间戳对齐。这可能涉及到将时间戳转换为统一的格式（如UNIX时间戳或标准日期时间格式），并确保两个数据集覆盖相同的时间段。
1.3 数据合并
根据时间戳将家庭用电量数据和天气数据合并为一个数据集。每个时间点的数据应包括用电量和所有天气参数。
2. 特征工程
特征选择：基于数据理解，选择对聚类可能有影响的特征，例如，体感温度、湿度、风速可能会对用电量有直接影响。
特征转换：将所有特征标准化或归一化，以确保它们在相同的尺度上进行比较。
3. 相似性度量与聚类算法选择
相似性度量：考虑使用动态时间弯曲（DTW）作为相似性度量，因为它能够有效处理时间序列之间的时间偏移和伸缩。
聚类算法：可以考虑使用K-均值聚类（特别是如果使用DTW，那么是DTW的变体，如K-Shape聚类），或者层次聚类算法，后者不需要预先指定聚类数目。
4. 聚类执行
执行聚类算法，将时间序列数据分组为不同的聚类。这些聚类可能基于用电行为和天气条件的相似模式。
5. 聚类结果分析与解释
分析聚类结果：检查每个聚类的特征，如平均用电量、平均气温、平均湿度等，以理解不同聚类代表的用电行为和天气条件的模式。
结果可视化：使用时间序列图、雷达图或热图来展示聚类结果，这有助于直观理解不同聚类之间的差异。
6. 后续步骤
根据聚类结果，可以进一步分析特定天气条件下的用电模式，或者识别特定的用电行为模式对应的天气条件，这对于能源需求预测和优化能源供应具有重要意义。


## 数据预处理

1. 家庭用电量数据预处理
1.1 转换时间粒度
由于家庭用电量数据是以半小时为单位，而天气数据是以小时为单位，需要将家庭用电量数据转换为小时单位。这可以通过计算每小时内两个半小时用电量的平均值来实现。

In [136]:
# 先复制原数据中的非用电量数据列
filtered_big_hhblock_one_hourly = filtered_big_hhblock[['LCLid', 'day']].copy()

# 计算每小时的平均用电量，并将结果添加到新DataFrame中
for i in range(24):
    filtered_big_hhblock_one_hourly[f'hh_{i}'] = filtered_big_hhblock[[f'hh_{2 * i}', f'hh_{2 * i + 1}']].sum(axis=1)

In [137]:
filtered_big_hhblock_one_hourly.head(1000)

,LCLid,day,hh_0,hh_1,hh_2,hh_3,hh_4,hh_5,hh_6,hh_7,...,hh_14,hh_15,hh_16,hh_17,hh_18,hh_19,hh_20,hh_21,hh_22,hh_23
0,MAC000002,2012-10-13,0.532,0.531,0.347,0.280,0.276,0.275,0.283,0.283,...,0.348,0.369,0.308,0.406,0.648,1.196,0.506,0.463,0.423,0.509
1,MAC000002,2012-10-14,0.428,0.314,0.208,0.206,0.199,0.196,0.197,0.195,...,0.280,0.231,0.256,0.464,1.471,2.031,1.566,1.223,0.441,0.437
2,MAC000002,2012-10-15,0.289,0.224,0.202,0.204,0.202,0.200,0.199,0.246,...,0.204,0.203,0.360,0.955,0.482,1.413,0.483,0.594,0.535,0.478
3,MAC000002,2012-10-16,0.474,0.311,0.205,0.203,0.196,0.195,0.195,0.194,...,0.207,0.206,0.368,0.488,0.713,1.138,0.420,0.414,0.401,0.359
4,MAC000002,2012-10-17,0.368,0.324,0.218,0.202,0.199,0.198,0.197,0.191,...,0.208,0.203,0.363,0.348,0.475,0.298,0.438,0.642,0.604,0.544
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,MAC000246,2013-04-15,1.466,0.397,0.105,0.132,0.137,0.164,0.808,0.509,...,0.582,0.523,0.514,0.663,0.840,2.851,4.091,2.248,1.317,1.544
996,MAC000246,2013-04-16,0.735,1.441,0.326,0.101,0.087,0.114,1.037,1.573,...,0.108,0.087,0.370,0.326,0.426,3.371,5.093,3.478,1.688,1.437
997,MAC000246,2013-04-17,1.376,0.517,0.102,0.082,0.103,0.124,0.494,0.843,...,0.554,0.239,0.208,0.377,2.036,1.198,2.194,1.084,1.537,0.404
998,MAC000246,2013-04-18,0.254,0.512,0.897,0.577,0.088,0.174,0.858,0.766,...,0.637,0.539,0.289,0.269,0.445,1.310,4.080,2.916,2.059,0.976


In [138]:
filtered_big_hhblock_one_hourly.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3439414 entries, 0 to 3469351
Data columns (total 26 columns):
 #   Column  Dtype         
---  ------  -----         
 0   LCLid   category      
 1   day     datetime64[ns]
 2   hh_0    float64       
 3   hh_1    float64       
 4   hh_2    float64       
 5   hh_3    float64       
 6   hh_4    float64       
 7   hh_5    float64       
 8   hh_6    float64       
 9   hh_7    float64       
 10  hh_8    float64       
 11  hh_9    float64       
 12  hh_10   float64       
 13  hh_11   float64       
 14  hh_12   float64       
 15  hh_13   float64       
 16  hh_14   float64       
 17  hh_15   float64       
 18  hh_16   float64       
 19  hh_17   float64       
 20  hh_18   float64       
 21  hh_19   float64       
 22  hh_20   float64       
 23  hh_21   float64       
 24  hh_22   float64       
 25  hh_23   float64       
dtypes: category(1), datetime64[ns](1), float64(24)
memory usage: 689.0 MB


2. 数据合并
2.1 时间格式转换
根据day和time列将两个数据集合并。这可能需要将家庭用电量数据的day列转换为与天气数据中time列相同的日期加小时的格式。

In [139]:
@memory.cache
def turn_wide_to_long(filtered_big_hhblock_one_hourly):
    # 转换LCLid列为分类数据类型
    filtered_big_hhblock_one_hourly['LCLid'] = filtered_big_hhblock_one_hourly['LCLid'].astype('category')
    
    # 转换Pandas DataFrame为Dask DataFrame
    ddf = dd.from_pandas(filtered_big_hhblock_one_hourly, npartitions=10)  # 根据数据大小和内存情况调整分区数

    # 先将Dask DataFrame转换为Pandas DataFrame进行melt操作
    pdf = ddf.compute()  # 将Dask DataFrame转换回Pandas DataFrame

    # 使用Pandas的melt函数进行操作
    pdf_melted = pdf.melt(id_vars=['LCLid', 'day'], var_name='hour', value_name='energy')

    # 转换 'hour' 列，从 'hh_X' 提取小时数，并将其转换为整数
    pdf_melted['hour'] = pdf_melted['hour'].str.extract('(\d+)').astype(int)

    # 假设 'day' 已经是 datetime 类型，如果不是，先转换它
    # 将 'day' 和 'hour' 合并成一个 datetime 列
    pdf_melted['day-hour'] = pd.to_datetime(pdf_melted['day']) + pd.to_timedelta(pdf_melted['hour'], unit='h')

    # 删除不再需要的列
    pdf_melted.drop(['day', 'hour'], axis=1, inplace=True)

    # 再将处理后的Pandas DataFrame转换回Dask DataFrame
    ddf_melted = dd.from_pandas(pdf_melted, npartitions=10)

    # 设置进度条
    pbar = ProgressBar()
    pbar.register()

    df_result = ddf_melted.compute()  # Progress bar will show up here

    # 注销进度条
    pbar.unregister()

    # 现在df_result是一个Pandas DataFrame，包含了处理后的数据

    df_result.to_parquet('filtered_big_hhblock_one_hourly_one_row.parquet')

    return df_result

In [140]:
# %%time
# if os.path.exists('filtered_big_hhblock_one_hourly_one_row.parquet'):
#     filtered_big_hhblock_one_hourly_one_row = pd.read_parquet('filtered_big_hhblock_one_hourly_one_row.parquet')
# else:
#     filtered_big_hhblock_one_hourly_one_row = turn_wide_to_long(filtered_big_hhblock_one_hourly)

In [141]:
%%time
filtered_big_hhblock_one_hourly_one_row = turn_wide_to_long(filtered_big_hhblock_one_hourly)

CPU times: total: 328 ms
Wall time: 1.52 s


In [142]:
filtered_big_hhblock_one_hourly_one_row.head(1000)

,LCLid,energy,day-hour
0,MAC000002,0.532,2012-10-13
1,MAC000002,0.428,2012-10-14
2,MAC000002,0.289,2012-10-15
3,MAC000002,0.474,2012-10-16
4,MAC000002,0.368,2012-10-17
...,...,...,...
995,MAC000246,1.466,2013-04-15
996,MAC000246,0.735,2013-04-16
997,MAC000246,1.376,2013-04-17
998,MAC000246,0.254,2013-04-18


2.2 数据合并，将转换格式后的家庭用电量数据和天气数据合并为一个数据集

In [143]:
# sampled_filtered_big_hhblock_one_hourly_one_row = filtered_big_hhblock_one_hourly_one_row.sample(frac=0.001, random_state=0)

In [144]:
# sampled_filtered_big_hhblock_one_hourly_one_row

In [145]:
weather_hourly_darksky.head(1000)

,visibility,windBearing,temperature,time,dewPoint,pressure,apparentTemperature,windSpeed,precipType,icon,humidity,summary
0,5.97,104,10.24,2011-11-11 00:00:00,8.86,1016.76,10.24,2.77,rain,partly-cloudy-night,0.91,Partly Cloudy
1,4.88,99,9.76,2011-11-11 01:00:00,8.83,1016.63,8.24,2.95,rain,partly-cloudy-night,0.94,Partly Cloudy
2,3.70,98,9.46,2011-11-11 02:00:00,8.79,1016.36,7.76,3.17,rain,partly-cloudy-night,0.96,Partly Cloudy
3,3.12,99,9.23,2011-11-11 03:00:00,8.63,1016.28,7.44,3.25,rain,fog,0.96,Foggy
4,1.85,111,9.26,2011-11-11 04:00:00,9.21,1015.98,7.24,3.70,rain,fog,1.00,Foggy
...,...,...,...,...,...,...,...,...,...,...,...,...
995,13.07,252,11.63,2011-12-31 11:00:00,9.73,1010.91,11.63,5.24,rain,partly-cloudy-day,0.88,Mostly Cloudy
996,13.52,251,11.88,2011-12-31 12:00:00,9.77,1010.65,11.88,6.08,rain,partly-cloudy-day,0.87,Mostly Cloudy
997,13.07,252,12.16,2011-12-31 13:00:00,10.09,1010.47,12.16,6.00,rain,partly-cloudy-day,0.87,Mostly Cloudy
998,12.89,251,12.17,2011-12-31 14:00:00,9.74,1010.15,12.17,5.98,rain,partly-cloudy-day,0.85,Mostly Cloudy


In [146]:
weather_hourly_darksky.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21165 entries, 0 to 21164
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   visibility           21165 non-null  float64       
 1   windBearing          21165 non-null  int64         
 2   temperature          21165 non-null  float64       
 3   time                 21165 non-null  datetime64[ns]
 4   dewPoint             21165 non-null  float64       
 5   pressure             21165 non-null  float64       
 6   apparentTemperature  21165 non-null  float64       
 7   windSpeed            21165 non-null  float64       
 8   precipType           21165 non-null  object        
 9   icon                 21165 non-null  object        
 10  humidity             21165 non-null  float64       
 11  summary              21165 non-null  object        
dtypes: datetime64[ns](1), float64(7), int64(1), object(3)
memory usage: 1.9+ MB


In [147]:
@memory.cache
def combine_data_with_mapping(filtered_big_hhblock_one_hourly_one_row, weather_hourly_darksky):
    # 初始化一个字典来保存转换映射
    mapping_dict = {}

    # 转换为dask dataframe
    ddf_weather = dd.from_pandas(weather_hourly_darksky, npartitions=10)

    # 首先全局计算出现频率并生成映射字典
    for col in ['precipType', 'icon', 'summary']:
        if col in ddf_weather.columns:
            # 使用dask计算每个值的出现次数
            frequencies = ddf_weather[col].value_counts().compute()
            # 根据频率排序，生成映射
            mapping = {value: i for i, value in enumerate(frequencies.sort_values(ascending=False).index)}
            mapping_dict[col] = mapping

            # 应用映射到每个分区，并提供meta参数以避免警告
        ddf_weather[col] = ddf_weather[col].map(mapping, meta=('x', 'int64')).astype('int64')
        
    # 使用Dask进行数据合并处理
    ddf_result = dd.from_pandas(filtered_big_hhblock_one_hourly_one_row, npartitions=10)

    combined_ddf = dd.merge(ddf_result, ddf_weather, left_on='day-hour', right_on='time', how='inner')

    # 设置进度条
    pbar = ProgressBar()
    pbar.register()

    # 计算合并后的Dask DataFrame，转换为Pandas DataFrame
    combined_pd = combined_ddf.compute()

    # 注销进度条
    pbar.unregister()

    combined_pd.drop(['time'], axis=1, inplace=True)

    combined_pd.to_parquet('combined_hourly_weather.parquet')

    # 返回合并后的DataFrame和映射字典
    return combined_pd, mapping_dict


In [148]:
# %%time
# if os.path.exists('combined_hourly_weather.parquet'):
#     print('Reading combined_hourly_weather.parquet...')
#     combined_hourly_weather = pd.read_parquet('combined_hourly_weather.parquet')
# else:
#     print('Combining data...')
#     combined_hourly_weather, mapping_dict = combine_data_with_mapping(filtered_big_hhblock_one_hourly_one_row, weather_hourly_darksky)

In [149]:
combined_hourly_weather, mapping_dict = combine_data_with_mapping(filtered_big_hhblock_one_hourly_one_row, weather_hourly_darksky)

In [150]:
combined_hourly_weather.head(1000)

,LCLid,energy,day-hour,visibility,windBearing,temperature,dewPoint,pressure,apparentTemperature,windSpeed,precipType,icon,humidity,summary
0,MAC002441,2.420,2013-05-13 07:00:00,11.84,257,11.39,6.88,1012.67,11.39,6.21,0,0,0.74,1
1,MAC002441,3.030,2013-05-14 07:00:00,12.26,234,8.76,4.53,1007.48,6.27,4.44,0,0,0.75,1
2,MAC002441,2.287,2013-05-16 07:00:00,7.92,182,7.81,6.52,1000.54,7.81,0.57,0,3,0.92,2
3,MAC002441,3.233,2013-05-28 07:00:00,4.47,58,10.41,9.40,1001.00,10.41,2.45,0,5,0.93,3
4,MAC002441,0.639,2013-06-03 07:00:00,13.89,39,12.34,6.36,1032.62,12.34,2.19,0,0,0.67,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,MAC000030,0.467,2013-03-28 07:00:00,7.61,17,-0.35,-2.32,1011.20,-3.61,2.67,1,3,0.87,2
996,MAC000030,0.162,2013-04-18 07:00:00,13.52,232,10.17,3.54,1011.44,10.17,9.05,0,4,0.63,6
997,MAC000030,0.424,2013-04-23 07:00:00,13.36,273,11.49,8.26,1018.52,11.49,3.91,0,0,0.81,0
998,MAC000030,0.316,2013-04-27 07:00:00,12.96,350,5.27,1.89,1013.80,2.35,3.73,0,0,0.79,0


In [151]:
display(mapping_dict)

{'precipType': {'rain': 0, 'snow': 1},
 'icon': {'partly-cloudy-day': 0,
  'partly-cloudy-night': 1,
  'clear-night': 2,
  'clear-day': 3,
  'wind': 4,
  'cloudy': 5,
  'fog': 6},
 'summary': {'Partly Cloudy': 0,
  'Mostly Cloudy': 1,
  'Clear': 2,
  'Overcast': 3,
  'Foggy': 4,
  'Breezy and Mostly Cloudy': 5,
  'Breezy and Partly Cloudy': 6,
  'Breezy': 7,
  'Breezy and Overcast': 8,
  'Windy and Mostly Cloudy': 9,
  'Windy': 10,
  'Windy and Overcast': 11,
  'Windy and Partly Cloudy': 12}}

In [152]:
combined_hourly_weather.head(1000)

,LCLid,energy,day-hour,visibility,windBearing,temperature,dewPoint,pressure,apparentTemperature,windSpeed,precipType,icon,humidity,summary
0,MAC002441,2.420,2013-05-13 07:00:00,11.84,257,11.39,6.88,1012.67,11.39,6.21,0,0,0.74,1
1,MAC002441,3.030,2013-05-14 07:00:00,12.26,234,8.76,4.53,1007.48,6.27,4.44,0,0,0.75,1
2,MAC002441,2.287,2013-05-16 07:00:00,7.92,182,7.81,6.52,1000.54,7.81,0.57,0,3,0.92,2
3,MAC002441,3.233,2013-05-28 07:00:00,4.47,58,10.41,9.40,1001.00,10.41,2.45,0,5,0.93,3
4,MAC002441,0.639,2013-06-03 07:00:00,13.89,39,12.34,6.36,1032.62,12.34,2.19,0,0,0.67,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,MAC000030,0.467,2013-03-28 07:00:00,7.61,17,-0.35,-2.32,1011.20,-3.61,2.67,1,3,0.87,2
996,MAC000030,0.162,2013-04-18 07:00:00,13.52,232,10.17,3.54,1011.44,10.17,9.05,0,4,0.63,6
997,MAC000030,0.424,2013-04-23 07:00:00,13.36,273,11.49,8.26,1018.52,11.49,3.91,0,0,0.81,0
998,MAC000030,0.316,2013-04-27 07:00:00,12.96,350,5.27,1.89,1013.80,2.35,3.73,0,0,0.79,0


2.3 缺失值处理

In [157]:
missing_values_count = combined_hourly_weather.isnull().sum()
missing_values_count

LCLid                  0
energy                 0
day-hour               0
visibility             0
windBearing            0
temperature            0
dewPoint               0
pressure               0
apparentTemperature    0
windSpeed              0
precipType             0
icon                   0
humidity               0
summary                0
hour                   0
day                    0
day_of_week            0
month                  0
quarter                0
year                   0
hour_sin               0
hour_cos               0
day_sin                0
day_cos                0
dtype: int64

2.4 数据标准化

2.4.1 特征列生成

将整体的时间转换为拆分的时间特征，以便更好地表示时间的周期性。
'day-hour' -> 'hour', 'day', 'day_of_week', 'month', 'quarter', 'year'
再将这些周期性时间特征转换为循环特征，以便更好地在模型中表示时间的周期性。
'hour' -> 'hour_sin', 'hour_cos'
'day' -> 'day_sin', 'day_cos'
'day_of_week' -> 'day_of_week_sin', 'day_of_week_cos'
'month' -> 'month_sin', 'month_cos'
'quarter' -> 'quarter_sin', 'quarter_cos'
'year' -> 'year_sin', 'year_cos'

In [158]:
@memory.cache
def add_cyclic_features(df):
    # 从'day-hour'列提取周期性特征
    df['hour'] = df['day-hour'].dt.hour
    df['day'] = df['day-hour'].dt.day
    df['day_of_week'] = df['day-hour'].dt.dayofweek
    df['month'] = df['day-hour'].dt.month
    df['quarter'] = df['day-hour'].dt.quarter
    df['year'] = df['day-hour'].dt.year

    # 转换小时特征为循环特征
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

    # # 转换天为循环特征
    df['day_sin'] = np.sin(2 * np.pi * df['day'] / 31)
    df['day_cos'] = np.cos(2 * np.pi * df['day'] / 31)

    # 转换星期特征为循环特征
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

    # 转换月份特征为循环特征
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    # 转换季度特征为循环特征
    df['quarter_sin'] = np.sin(2 * np.pi * df['quarter'] / 4)
    df['quarter_cos'] = np.cos(2 * np.pi * df['quarter'] / 4)

    # 转换年份特征为循环特征
    df['year_sin'] = np.sin(2 * np.pi * df['year'] / 365)
    df['year_cos'] = np.cos(2 * np.pi * df['year'] / 365)
    
    # 删除原始时间特征列
    df.drop(['day-hour', 'hour', 'day', 'day_of_week', 'month', 'quarter', 'year'], axis=1, inplace=True)
    
    return df

In [159]:
%%time
combined_hourly_weather_all_features = add_cyclic_features(combined_hourly_weather)

CPU times: total: 31.1 s
Wall time: 2min 5s


<timed exec>:1: UserWarning: Persisting input arguments took 62.69s to run.If this happens often in your code, it can cause performance problems (results will be correct in all cases). The reason for this is probably some large input arguments for a wrapped function.


In [160]:
combined_hourly_weather_all_features.head(1000)

,LCLid,energy,visibility,windBearing,temperature,dewPoint,pressure,apparentTemperature,windSpeed,precipType,...,day_sin,day_cos,day_of_week_sin,day_of_week_cos,month_sin,month_cos,quarter_sin,quarter_cos,year_sin,year_cos
0,MAC002441,2.420,11.84,257,11.39,6.88,1012.67,11.39,6.21,0,...,0.485302,-0.874347,0.000000,1.000000,5.000000e-01,-8.660254e-01,1.224647e-16,-1.000000e+00,-0.094537,-0.995521
1,MAC002441,3.030,12.26,234,8.76,4.53,1007.48,6.27,4.44,0,...,0.299363,-0.954139,0.781831,0.623490,5.000000e-01,-8.660254e-01,1.224647e-16,-1.000000e+00,-0.094537,-0.995521
2,MAC002441,2.287,7.92,182,7.81,6.52,1000.54,7.81,0.57,0,...,-0.101168,-0.994869,0.433884,-0.900969,5.000000e-01,-8.660254e-01,1.224647e-16,-1.000000e+00,-0.094537,-0.995521
3,MAC002441,3.233,4.47,58,10.41,9.40,1001.00,10.41,2.45,0,...,-0.571268,0.820763,0.781831,0.623490,5.000000e-01,-8.660254e-01,1.224647e-16,-1.000000e+00,-0.094537,-0.995521
4,MAC002441,0.639,13.89,39,12.34,6.36,1032.62,12.34,2.19,0,...,0.571268,0.820763,0.000000,1.000000,1.224647e-16,-1.000000e+00,1.224647e-16,-1.000000e+00,-0.094537,-0.995521
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,MAC000030,0.467,7.61,17,-0.35,-2.32,1011.20,-3.61,2.67,1,...,-0.571268,0.820763,0.433884,-0.900969,1.000000e+00,6.123234e-17,1.000000e+00,6.123234e-17,-0.094537,-0.995521
996,MAC000030,0.162,13.52,232,10.17,3.54,1011.44,10.17,9.05,0,...,-0.485302,-0.874347,0.433884,-0.900969,8.660254e-01,-5.000000e-01,1.224647e-16,-1.000000e+00,-0.094537,-0.995521
997,MAC000030,0.424,13.36,273,11.49,8.26,1018.52,11.49,3.91,0,...,-0.998717,-0.050649,0.781831,0.623490,8.660254e-01,-5.000000e-01,1.224647e-16,-1.000000e+00,-0.094537,-0.995521
998,MAC000030,0.316,12.96,350,5.27,1.89,1013.80,2.35,3.73,0,...,-0.724793,0.688967,-0.974928,-0.222521,8.660254e-01,-5.000000e-01,1.224647e-16,-1.000000e+00,-0.094537,-0.995521


In [161]:
combined_hourly_weather_all_features.columns

Index(['LCLid', 'energy', 'visibility', 'windBearing', 'temperature',
       'dewPoint', 'pressure', 'apparentTemperature', 'windSpeed',
       'precipType', 'icon', 'humidity', 'summary', 'hour_sin', 'hour_cos',
       'day_sin', 'day_cos', 'day_of_week_sin', 'day_of_week_cos', 'month_sin',
       'month_cos', 'quarter_sin', 'quarter_cos', 'year_sin', 'year_cos'],
      dtype='object')

2.4.2 计算相关系数

In [162]:
def compute_correlation(series_energy, column, series_column):
    # 确保序列是NumPy数组格式
    series_energy_np = np.array(series_energy)
    series_column_np = np.array(series_column)

    # 计算Pearson、Spearman和Kendall相关性
    pearson_corr = series_energy.corr(series_column, method='pearson')
    spearman_corr = series_energy.corr(series_column, method='spearman')
    kendall_corr = series_energy.corr(series_column, method='kendall')

    # 计算DTW距离
    dtw_distance = dtw.distance(series_energy_np, series_column_np)

    return column, pearson_corr, spearman_corr, kendall_corr, dtw_distance

In [163]:
sampled_combined_hourly_weather_all_features = combined_hourly_weather_all_features.sample(frac=0.0001, random_state=0)
sampled_combined_hourly_weather_all_features

,LCLid,energy,visibility,windBearing,temperature,dewPoint,pressure,apparentTemperature,windSpeed,precipType,...,day_sin,day_cos,day_of_week_sin,day_of_week_cos,month_sin,month_cos,quarter_sin,quarter_cos,year_sin,year_cos
8055381,MAC003438,0.620,7.11,48,0.23,-1.81,997.66,-4.41,4.64,1,...,-0.651372,-0.758758,-0.974928,-0.222521,5.000000e-01,8.660254e-01,1.000000e+00,6.123234e-17,-0.094537,-0.995521
2755035,MAC001708,0.282,3.12,68,6.46,5.90,994.64,2.98,5.42,0,...,0.790776,-0.612106,0.433884,-0.900969,8.660254e-01,-5.000000e-01,1.224647e-16,-1.000000e+00,-0.094537,-0.995521
6859792,MAC003034,0.391,16.09,89,12.17,1.02,1021.93,12.17,3.29,0,...,0.394356,0.918958,0.433884,-0.900969,5.000000e-01,-8.660254e-01,1.224647e-16,-1.000000e+00,-0.094537,-0.995521
8184989,MAC001523,0.624,10.75,207,15.21,13.69,1007.04,15.21,2.20,0,...,0.937752,0.347305,-0.433884,-0.900969,-5.000000e-01,-8.660254e-01,-1.000000e+00,-1.836970e-16,-0.077386,-0.997001
331406,MAC003270,0.367,12.76,21,3.28,-4.72,1009.17,-1.27,5.96,0,...,0.651372,-0.758758,0.781831,0.623490,1.000000e+00,6.123234e-17,1.000000e+00,6.123234e-17,-0.094537,-0.995521
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1588242,MAC001156,0.312,9.46,220,12.62,11.02,1002.66,12.62,9.14,0,...,-0.394356,0.918958,0.781831,0.623490,5.000000e-01,8.660254e-01,1.000000e+00,6.123234e-17,-0.094537,-0.995521
918361,MAC004148,0.240,2.49,222,7.62,7.09,1013.44,6.76,1.65,0,...,-0.937752,0.347305,-0.974928,-0.222521,5.000000e-01,8.660254e-01,1.000000e+00,6.123234e-17,-0.111659,-0.993747
7415581,MAC000430,0.070,5.99,329,1.26,-0.86,1029.38,1.26,1.32,0,...,0.790776,-0.612106,0.781831,0.623490,-2.449294e-16,1.000000e+00,-2.449294e-16,1.000000e+00,-0.077386,-0.997001
3334751,MAC001592,0.364,11.67,43,2.74,-3.49,1014.33,-2.32,6.80,0,...,0.724793,0.688967,0.433884,-0.900969,8.660254e-01,-5.000000e-01,1.224647e-16,-1.000000e+00,-0.094537,-0.995521


In [164]:
series_energy = sampled_combined_hourly_weather_all_features['energy']

In [165]:
df_dropped = sampled_combined_hourly_weather_all_features.drop(['LCLid', 'energy'], axis=1)

In [166]:
results = Parallel(n_jobs=32)(delayed(compute_correlation)(series_energy, column, df_dropped[column]) for column in df_dropped.columns)

In [167]:
correlation_df = pd.DataFrame(results, columns=['Column', 'Pearson', 'Spearman', 'Kendall', 'DTW']).set_index('Column')

In [168]:
correlation_df

,Pearson,Spearman,Kendall,DTW
Column,,,,
visibility,-0.006904,0.016826,0.011255,443.862792
windBearing,-0.006027,-0.006485,-0.004370,18906.127867
temperature,-0.083022,-0.054545,-0.036281,626.412410
dewPoint,-0.097926,-0.078054,-0.051959,466.723752
pressure,-0.011370,-0.005006,-0.003272,91988.135263
apparentTemperature,-0.086513,-0.055480,-0.036885,683.244370
windSpeed,0.013820,0.030065,0.020107,190.000330
precipType,0.034258,0.018888,0.015430,48.576321
icon,-0.012922,-0.050057,-0.036609,120.986451


In [169]:
print(correlation_df)

                      Pearson  Spearman   Kendall           DTW
Column                                                         
visibility          -0.006904  0.016826  0.011255    443.862792
windBearing         -0.006027 -0.006485 -0.004370  18906.127867
temperature         -0.083022 -0.054545 -0.036281    626.412410
dewPoint            -0.097926 -0.078054 -0.051959    466.723752
pressure            -0.011370 -0.005006 -0.003272  91988.135263
apparentTemperature -0.086513 -0.055480 -0.036885    683.244370
windSpeed            0.013820  0.030065  0.020107    190.000330
precipType           0.034258  0.018888  0.015430     48.576321
icon                -0.012922 -0.050057 -0.036609    120.986451
humidity            -0.011574 -0.054400 -0.036372     43.448652
summary             -0.019059 -0.054705 -0.040448    116.506113
hour_sin            -0.163699 -0.234208 -0.160950     64.558582
hour_cos            -0.029295 -0.063006 -0.042857     64.119046
day_sin              0.004296 -0.001687 

某次frac=0.0001(8254 rows)的结果：
                      Pearson  Spearman   Kendall           DTW
Column                                                         
visibility          -0.006904  0.016826  0.011255    443.862792
windBearing         -0.006027 -0.006485 -0.004370  18906.127867
temperature         -0.083022 -0.054545 -0.036281    626.412410
dewPoint            -0.097926 -0.078054 -0.051959    466.723752
pressure            -0.011370 -0.005006 -0.003272  91988.135263
apparentTemperature -0.086513 -0.055480 -0.036885    683.244370
windSpeed            0.013820  0.030065  0.020107    190.000330
precipType           0.034258  0.018888  0.015430     48.576321
icon                -0.012922 -0.050057 -0.036609    120.986451
humidity            -0.011574 -0.054400 -0.036372     43.448652
summary             -0.019059 -0.054705 -0.040448    116.506113
hour_sin            -0.163699 -0.234208 -0.160950     64.558582
hour_cos            -0.029295 -0.063006 -0.042857     64.119046
day_sin              0.004296 -0.001687 -0.001163     64.604965
day_cos             -0.007818  0.000127  0.000092     63.945644
day_of_week_sin     -0.008611 -0.005341 -0.003856     63.926037
day_of_week_cos      0.024850  0.011692  0.008394     65.651662
month_sin            0.069973  0.042131  0.029486     65.454581
month_cos            0.084735  0.067386  0.047056     63.750382
quarter_sin          0.095618  0.057290  0.042831     64.398062
quarter_cos          0.040359  0.061399  0.045760     60.926613
year_sin            -0.024352 -0.008327 -0.006598     65.629302
year_cos             0.024858  0.008327  0.006598    137.013980



In [ ]:
sampled_combined_hourly_weather_all_features.columns

In [ ]:
columns_morethan0 = []
for row in correlation_df.itertuples():
    if (row.Pearson > 0) or (row.Spearman > 0) or (row.Kendall > 0) or (row.DTW < 100):
        columns_morethan0.append(row.Index)
        
print(columns_morethan0)

In [ ]:
# 将DataFrame的列名和columns_morethan0转换为集合
columns_set = set(sampled_combined_hourly_weather_all_features.columns)
columns_morethan0_set = set(columns_morethan0)

# 检查是否所有列都在columns_morethan0中
if not columns_set.issubset(columns_morethan0_set):
    missing_columns = columns_set - columns_morethan0_set
    print("These columns are not in columns_morethan0:", missing_columns)


sin和cos只有一个循环特征相关性高的可能原因：
- 非对称影响：如果目标变量对时间的敏感度是非对称的，比如说，某些现象或行为在一天中的某些时间段更为频繁或强烈，这种非对称性可能会导致sin或cos中的一个与目标变量有更高的相关性。例如，如果energy消耗在夜间减少但在白天变化不大，那么sin函数（代表了一天中的变化）可能与energy有更高的相关性。
- 数据的内在特性：数据集中的某些特性可能与特定的循环特征更加对齐。例如，如果能量消耗主要在某个特定的时间段内变化显著，那么这种模式可能会更多地反映在sin或cos值的变化上。

根据结果，排除相关性较低的特征：'dewPoint', 'pressure', 'icon', 'summary', 同时，虽然'temperature', 'apparentTemperature'的各项相关性都不高，但是由于它们是气象数据中的重要特征，因此保留。

In [ ]:
cluster_columns_use = [
    'LCLid', 'energy',
    'hour_sin', 'hour_cos',
    'day_sin', 'day_cos',
    'day_of_week_sin', 'day_of_week_cos',
    'month_sin', 'month_cos',
    'quarter_sin', 'quarter_cos',
    'year_sin', 'year_cos',
    'visibility', 
    'windSpeed', 
    'precipType',
    'humidity',
    'temperature', 'apparentTemperature',]
cluster_columns_not_use = ['dewPoint', 'pressure', 'icon', 'summary', 'windBearing' ]

2.4.3 数值型数据标准化

In [ ]:
df_cluster = combined_hourly_weather_all_features[cluster_columns_use]
df_cluster.head(1000)

In [ ]:
def std_scaler(df):
    # 初始化StandardScaler
    scaler = StandardScaler()
    
    norminal_columns = ['energy', 'visibility', 'windSpeed', 'precipType', 'humidity', 'temperature', 'apparentTemperature']
    
    df_copy = df.copy()
    
    # 直接在原地修改，节省空间（注意：这会修改原始DataFrame）
    df_copy[norminal_columns] = scaler.fit_transform(df_copy[norminal_columns])
    
    return df_copy

In [ ]:
df_cluster_normalised = std_scaler(df_cluster)

In [ ]:
df_cluster_normalised.head(1000)

In [ ]:
df_cluster_normalised.info()

分别使用K-均值聚类

In [ ]:
features = {
    'hour': ['hour', 'hour_sin', 'hour_cos'],
    'day': ['day', 'day_sin', 'day_cos'],
    'day_of_week': ['day_of_week', 'day_of_week_sin', 'day_of_week_cos'],
    'month': ['month', 'month_sin', 'month_cos'],
    'quarter': ['quarter', 'quarter_sin', 'quarter_cos'],
    'year': ['year', 'year_sin', 'year_cos'],
    'visibility': 'visibility',
    'windSpeed': 'windSpeed',
    'precipType': 'precipType',
    'humidity':'humidity',
    'weather': ['visibility',  'windSpeed', 'precipType', 'humidity'],
    'temperature': ['temperature', 'apparentTemperature']}

In [ ]:
@memory.cache
def cluster(judge, features, features_name):
    # 执行K-均值聚类
    kmeans = KMeans(n_clusters=4, random_state=0, n_init=10).fit(judge[features])

    # 将聚类标签添加到原始DataFrame中
    judge[f'cluster_kmeans_{features_name}'] = kmeans.labels_

    return judge

In [ ]:
# 创建任务列表
tasks = [(sampled_judge, features[features_name], features_name) for features_name in features.keys()]

In [ ]:
# 使用tqdm处理任务列表
results = Parallel(n_jobs=16)(delayed(cluster)(*task) for task in tqdm(tasks, desc="Processing"))

In [ ]:
# judge = Parallel(n_jobs=-1)(delayed(cluster)(sampled_judge, features[features_name], features_name) for features_name in features.keys())

In [ ]:
sampled_judge

In [ ]:
results[0]

In [ ]:
judge = results[1]

评估聚类质量

In [ ]:
def evaluate_clustering(judge, features, features_name):
    # 计算轮廓系数
    score_kmeans = silhouette_score(features, judge[f'cluster_kmeans_{features_name}'], metric='euclidean')
    print(f'Silhouette Score (K-Means) for {features_name}: {score_kmeans:.2f}')

    score_dbscan = silhouette_score(features, judge[f'cluster_dbscan_{features_name}'], metric='euclidean')
    print(f'Silhouette Score (DBSCAN) for {features_name}: {score_dbscan:.2f}')


In [ ]:
evaluate_clustering(judge, features['all'], 'all')

In [ ]:
evaluate_clustering(judge, features['hour'], 'hour')

In [ ]:
evaluate_clustering(judge, features['description'], 'description')

In [ ]:
evaluate_clustering(judge, features['windSpeed'], 'windSpeed')

In [ ]:
evaluate_clustering(judge, features['year'], 'year')

In [ ]:
evaluate_clustering(judge, features['precipType'], 'precipType')

In [ ]:
evaluate_clustering(judge, features['day_of_week'], 'day_of_week')

In [ ]:
evaluate_clustering(judge, features['temperature'], 'temperature')

可视化

In [ ]:
# t-SNE降维
tsne = TSNE(n_components=2, verbose=1, perplexity=40, n_iter=300)
tsne_results = tsne.fit_transform(features['all'])


def plot_t_SNE(tsne_results, judge):
    # 设置画布，1行2列的子图
    fig, ax = plt.subplots(1, 2, figsize=(32, 10))  # 宽度翻倍以容纳两个子图

    # 第一个子图，绘制K-Means的t-SNE可视化结果
    sns.scatterplot(
        x=tsne_results[:, 0], y=tsne_results[:, 1],
        hue=judge[f'cluster_kmeans_all'],
        palette=sns.color_palette("hsv", 4),
        legend="full",
        alpha=0.3,
        ax=ax[0]  # 指定第一个子图
    )
    ax[0].set_title(f't-SNE visualization of K-Means Clustering for all features')

    # 第二个子图，绘制DBSCAN的t-SNE可视化结果
    sns.scatterplot(
        x=tsne_results[:, 0], y=tsne_results[:, 1],
        hue=judge[f'cluster_dbscan_all'],
        palette=sns.color_palette("hsv", len(np.unique(judge[f'cluster_dbscan_all']))),
        legend="full",
        alpha=0.3,
        ax=ax[1]  # 指定第二个子图
    )
    ax[1].set_title(f't-SNE visualization of DBSCAN Clustering for all features')

    # 显示整个画布
    plt.show()

In [ ]:
plot_t_SNE(tsne_results, judge)

In [ ]:
def plot_pairplot(judge, features, features_name):
    fig, ax = plt.subplots(1, 2, figsize=(32, 10))

    # 使用K-Means的聚类结果
    sns.pairplot(judge, vars=features[features_name], hue=f'cluster_kmeans_{features_name}', palette='bright')
    plt.title(f'K-Means Clustering for {features_name}')
    plt.show()

    # 使用DBSCAN的聚类结果
    sns.pairplot(judge, vars=features[features_name], hue=f'cluster_dbscan_{features_name}', palette='bright')
    plt.title(f'DBSCAN Clustering for {features_name}')
    plt.show()

In [ ]:
plot_pairplot(judge, features, 'hour')

In [ ]:
plot_pairplot(judge, features, 'description')

In [ ]:
plot_pairplot(judge, features, 'windSpeed')

In [ ]:
plot_pairplot(judge, features, 'year')

In [ ]:
plot_pairplot(judge, features, 'precipType')

In [ ]:
plot_pairplot(judge, features, 'day_of_week')

In [ ]:
plot_pairplot(judge, features, 'temperature')

随机森林

In [ ]:
# from sklearn.model_selection import train_test_split
# 
# # 假设你的DataFrame名为df
# X = judge[['temperature']]  # 选择温度作为特征变量
# y = judge['energy']  # 能耗作为目标变量
# 
# # 划分训练集和测试集
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# from sklearn.ensemble import RandomForestRegressor
# 
# # 初始化随机森林回归器
# rf = RandomForestRegressor(n_estimators=100, random_state=42)  # n_estimators是树的数量
# 
# # 训练模型
# rf.fit(X_train, y_train)


In [ ]:
# from sklearn.metrics import mean_squared_error
# 
# # 对测试集进行预测
# y_pred = rf.predict(X_test)
# 
# # 计算并打印均方误差(MSE)作为性能的评估
# mse = mean_squared_error(y_test, y_pred)
# print(f"Mean Squared Error: {mse}")


In [ ]:
# # 获取特征重要性
# importances = rf.feature_importances_
# print(f"Importance of temperature: {importances[0]}")


In [ ]:
# # 对一系列温度值进行预测以绘制曲线
# temperature_range = np.linspace(X_train.min(), X_train.max(), 100).reshape(-1, 1)
# energy_pred = rf.predict(temperature_range)
# 
# # 绘制散点图和预测曲线
# plt.scatter(X_train, y_train, color='gray', alpha=0.5, label='Actual')
# plt.plot(temperature_range, energy_pred, color='red', label='Prediction')
# plt.xlabel('Temperature')
# plt.ylabel('Energy Consumption')
# plt.title('Energy Consumption vs Temperature')
# plt.legend()
# plt.show()
